<div style="background:linear-gradient(120deg,#00553A 0%,#00704A 45%,#00A86A 100%);
            padding:34px 38px 30px 38px;border-radius:6px;color:#fff;
            font-family:Calibri,'Segoe UI',sans-serif">
  <div style="font-size:11px;font-weight:700;letter-spacing:3px;color:#F5C242;text-transform:uppercase">
    03 · DONNÉES NON TRADITIONNELLES &nbsp;·&nbsp; LABORATOIRE
  </div>
  <div style="font-size:38px;font-weight:700;line-height:1.12;margin-top:10px">
    Données ouvertes Ookla Speedtest &amp; WorldPop
  </div>
  <div style="font-size:17px;font-style:italic;color:#E6F6EE;margin-top:8px">
    Des carreaux de mesure participative à un indicateur infranational de connectivité pondéré par la population
  </div>
  <div style="height:5px;background:#F5C242;margin-top:24px;width:120px"></div>
  <div style="font-size:12.5px;color:#E6F6EE;margin-top:18px;line-height:1.6">
    Banque africaine de développement &nbsp;·&nbsp; UA STATAFRIC &nbsp;·&nbsp; Atelier technique STG17<br>
    <b>Emerging Issues, Emerging Practice</b> — Innover dans la chaîne de valeur de la donnée
  </div>
</div>

## Ce que fait ce notebook

Vous lui donnez **un code pays**. Il vous rend un **tableau de bord publié**.

Entre les deux, il déroule la chaîne de valeur complète telle qu'un INS la mettrait en production :

| Étape | Ce qui se passe |
|---|---|
| **Acquérir** | Interroge *à distance* les fichiers parquet mondiaux d'Ookla Open Data — seules les lignes couvrant votre pays sont téléchargées (élagage par intervalles de quadkeys, 100 à 500 fois moins de trafic qu'un téléchargement complet) |
| **Acquérir** | Récupère les limites administratives officielles (geoBoundaries) et la grille de population WorldPop |
| **Vérifier** | Exécute un **diagnostic de couverture** *avant* la conception du moindre indicateur : quelle part du territoire et de la population est réellement mesurée |
| **Intégrer** | Croise les carreaux de mesure de ~600 m avec la grille de population de 1 km et avec les limites ADM1/ADM2 |
| **Analyser** | Débits pondérés par la population, lacunes de couverture, fracture urbain–rural, fixe et mobile, évolutions trimestrielles, courbe de Lorenz de la connectivité |
| **Communiquer** | Cartes interactives, graphiques et un **tableau de bord HTML bilingue autonome**, publiable sur GitHub Pages |
| **Documenter** | Génère automatiquement le README, le bloc de sources et une **déclaration explicite des limites** — qui fait partie du livrable, et non d'un avertissement final |

## Objectifs pédagogiques

À l'issue de ce laboratoire, vous saurez :

1. Expliquer ce qu'**est** un carreau de performance Ookla, et ce qu'il **n'est pas** (une mesure participative auto-sélectionnée, jamais une carte de couverture).
2. Interroger efficacement un fichier parquet distant de plusieurs centaines de mégaoctets avec **DuckDB** et des prédicats d'intervalle sur le quadkey.
3. Décoder vous-même un **quadkey** en coordonnées géographiques, sans bibliothèque auxiliaire.
4. Pondérer une statistique de connectivité par la **population** plutôt que par le nombre de carreaux — et expliquer pourquoi les deux réponses diffèrent à ce point.
5. Produire et publier un produit analytique reproductible, avec une méthode, une licence et des limites documentées.

## Comment exécuter ce notebook

| Environnement | À faire |
|---|---|
| **Google Colab** | `Exécution → Tout exécuter`. Les paquets manquants s'installent automatiquement (≈ 2 min). |
| **Kaggle** | Activez **Internet** dans le panneau de droite (Settings → Internet), puis `Run All`. |
| **Local / JupyterLab** | Python ≥ 3.9. La cellule d'installation complète ce qui manque. |

**Durée attendue :** 4 à 9 minutes pour un petit pays (Rwanda, Togo, Djibouti), 10 à 25 minutes pour un grand (Nigeria, RDC, Égypte).
**Aucune clé d'API ni aucun compte ne sont nécessaires** — toutes les sources utilisées ici sont ouvertes.

> **Note sur la langue.** Ce notebook est rédigé en français ; le tableau de bord produit, lui, est **bilingue français / anglais** avec un sélecteur de langue. Une version anglaise du notebook existe également.

---

# 1 · Configuration

> **C'est la seule cellule que vous avez besoin de modifier.** Tout le reste en découle.

Remplacez `COUNTRY_ISO3` par n'importe quel code ISO 3166-1 alpha-3 et relancez le notebook :
`RWA`, `TUN`, `CIV`, `CMR`, `MOZ`, `SOM`, `NGA`, `KEN`, `SEN`, `MAR`, `ZAF`, `EGY`, `GHA`, `TGO`, `DZA`…

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  C O N F I G U R A T I O N   D U   L A B O R A T O I R E
# ══════════════════════════════════════════════════════════════════════════════

COUNTRY_ISO3 = "RWA"       # ISO-3166-1 alpha-3.  RWA = Rwanda

# --- Ookla -------------------------------------------------------------------
N_QUARTERS   = 4           # nombre de trimestres récents à récupérer (1 = le plus rapide)
SERVICES     = ["fixed", "mobile"]   # ["mobile"] seul : environ 2 fois plus rapide

# --- WorldPop ----------------------------------------------------------------
WORLDPOP_YEAR = 2020       # 2000-2020 disponibles pour le produit mondial à 1 km
WORLDPOP_RES  = "1km"      # "1km" (recommandé, rapide) ou "100m" (lourd, precis)

# --- Niveau administratif ----------------------------------------------------
ADMIN_LEVEL   = "ADM2"     # "ADM1" (regions/provinces) ou "ADM2" (districts)
                           # bascule automatiquement sur ADM1 si ADM2 est absent

# --- Seuils d'analyse ---------------------------------------------------
BROADBAND_MBPS   = 10      # seuil de référence UIT du haut débit utilisable
GOOD_SPEED_MBPS  = 25      # seuil de connectivité de bonne qualité
MIN_TESTS_TILE   = 1       # écarter les carreaux ayant moins de N tests dans le trimestre
URBAN_DENS_MIN   = 1500    # habitants / km2 -> urbain  (approximation inspirée de GHSL)
PERIURBAN_DENS_MIN = 300   # habitants / km2 -> périurbain

# --- Sorties ------------------------------------------------------------------
OUTPUT_DIR    = "outputs"  # tableau de bord, CSV, GeoJSON et README sont écrits ici
CACHE_DIR     = "cache"    # les fichiers téléchargés sont mis en cache ici (suppression sans risque)
MAX_MAP_TILES = 9000       # polygones sur la carte des carreaux (agrégés au-delà).
                           # Chaque langue porte sa propre copie de chaque carte : ce
                           # réglage pilote la taille du tableau de bord publié.

# ══════════════════════════════════════════════════════════════════════════════
print(f"Pays          : {COUNTRY_ISO3}")
print(f"Trimestres    : les {N_QUARTERS} derniers")
print(f"Services      : {', '.join(SERVICES)}")
print(f"Niveau admin  : {ADMIN_LEVEL}")
print(f"WorldPop      : {WORLDPOP_YEAR} @ {WORLDPOP_RES}")

# 2 · Environnement

La cellule ci-dessous détecte où vous vous trouvez (Colab / Kaggle / local) et n'installe **que ce
qui manque**. Elle est volontairement écrite avec `subprocess` plutôt qu'avec la commande magique
`%pip`, afin que le notebook s'exécute aussi comme un script ordinaire
(`jupyter nbconvert --execute`) — c'est ainsi qu'il tournera dans une chaîne d'intégration continue
une fois publié.

In [ ]:
import importlib, subprocess, sys, os, warnings
warnings.filterwarnings("ignore")

# --- où sommes-nous ? -----------------------------------------------------------
IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
ENV = "Google Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "Local / JupyterLab")

# Kaggle n'écrit que dans /kaggle/working
if IN_KAGGLE:
    OUTPUT_DIR = "/kaggle/working/" + OUTPUT_DIR
    CACHE_DIR  = "/kaggle/working/" + CACHE_DIR

REQUIRED = {           # nom d'import  ->  nom pip
    "requests":   "requests",
    "pandas":     "pandas",
    "numpy":      "numpy",
    "pyarrow":    "pyarrow",
    "duckdb":     "duckdb",
    "shapely":    "shapely",
    "geopandas":  "geopandas",
    "rasterio":   "rasterio",
    "folium":     "folium",
    "branca":     "branca",
    "plotly":     "plotly",
}

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
print(f"Environnement : {ENV}")
print(f"Python        : {sys.version.split()[0]}")

if missing:
    print(f"Installation  : {', '.join(missing)} …")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
    importlib.invalidate_caches()
    print("Installé.")
else:
    print("Dépendances   : toutes présentes ✓")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR,  exist_ok=True)
print(f"Sorties       : {os.path.abspath(OUTPUT_DIR)}")

In [ ]:
import json, math, time, io, textwrap, datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import duckdb
import geopandas as gpd
import rasterio
from rasterio.warp import transform_bounds
from shapely.geometry import box, Point
import folium
from folium.plugins import HeatMap, Fullscreen, MiniMap
import branca.colormap as cm
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from IPython.display import HTML, display, Markdown

# ══════════════════════════════════════════════════════════════════════════════
#  Palette institutionnelle inspirée de la BAD (cf. AfDB_Style_Guide_Presentations.md)
# ══════════════════════════════════════════════════════════════════════════════
GREEN, DEEP, FOREST = "#00A86A", "#00704A", "#00553A"
GOLD,  OCHRE        = "#F5C242", "#D49A00"
TEAL,  TERRA, BRICK = "#0E7C86", "#C4621D", "#B83B2E"
INK,   SLATE        = "#231F20", "#5E6964"
MIST,  MINT,  SAGE  = "#F4F7F5", "#E8F5EF", "#D5DED9"

RAMP    = ["#9ED9C0", "#7BCBA9", "#57BD92", "#33AF7C", "#10A06A", "#008A5B", "#00664A"]
CATCOL  = [GREEN, DEEP, TEAL, OCHRE, TERRA, BRICK]
FONT    = "Calibri, Segoe UI, Helvetica, Arial, sans-serif"

pio.templates["afdb"] = go.layout.Template(layout=dict(
    font=dict(family=FONT, size=13, color=INK),
    colorway=CATCOL,
    paper_bgcolor="white", plot_bgcolor="white",
    title=dict(font=dict(size=18, color=INK)),
    xaxis=dict(gridcolor="#E1E7E4", zeroline=False, linecolor=SAGE,
               tickfont=dict(color=SLATE, size=11)),
    yaxis=dict(gridcolor="#E1E7E4", zeroline=False, linecolor=SAGE,
               tickfont=dict(color=SLATE, size=11)),
    legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(size=11, color=SLATE)),
    margin=dict(l=60, r=30, t=70, b=55),
    hoverlabel=dict(font=dict(family=FONT, size=12), bgcolor="white",
                    bordercolor=SAGE),
))
pio.templates.default = "afdb"


# ══════════════════════════════════════════════════════════════════════════════
#  Petits utilitaires de présentation, utilisés dans tout le notebook
# ══════════════════════════════════════════════════════════════════════════════
def banner(kicker, title, body="", color=GREEN):
    display(HTML(f"""
    <div style="font-family:{FONT};border-left:4px solid {color};background:{MIST};
                padding:12px 16px;margin:6px 0 14px 0;border-radius:0 4px 4px 0">
      <div style="font-size:10.5px;font-weight:700;letter-spacing:2.5px;
                  text-transform:uppercase;color:{DEEP}">{kicker}</div>
      <div style="font-size:15px;font-weight:700;color:{INK};margin-top:3px">{title}</div>
      <div style="font-size:12.5px;color:{SLATE};margin-top:4px;line-height:1.55">{body}</div>
    </div>"""))

def callout(text, kind="info"):
    c = {"info": GREEN, "warn": OCHRE, "risk": BRICK}[kind]
    bg = {"info": MINT, "warn": "#FDF4E0", "risk": "#FBECEA"}[kind]
    icon = {"info": "●", "warn": "▲", "risk": "■"}[kind]
    display(HTML(f"""
    <div style="font-family:{FONT};background:{bg};border:1px solid {c};
                border-radius:5px;padding:11px 15px;margin:10px 0;font-size:12.5px;
                color:{INK};line-height:1.6"><b style="color:{c}">{icon}</b> &nbsp;{text}</div>"""))

def kpi_row(items):
    """items = [(libellé, valeur, unité, couleur), ...]"""
    cards = "".join(f"""
      <div style="flex:1;min-width:150px;background:white;border:1px solid {SAGE};
                  border-radius:6px;padding:14px 16px">
        <div style="font-size:9.5px;font-weight:700;letter-spacing:2px;
                    text-transform:uppercase;color:{SLATE}">{lab}</div>
        <div style="font-size:30px;font-weight:700;color:{col};line-height:1.15;
                    margin-top:5px">{val}<span style="font-size:13px;font-weight:600;
                    color:{SLATE};margin-left:3px">{unit}</span></div>
      </div>""" for lab, val, unit, col in items)
    display(HTML(f'<div style="display:flex;gap:11px;flex-wrap:wrap;'
                 f'font-family:{FONT};margin:10px 0 16px 0">{cards}</div>'))

def style_table(df, caption=""):
    """Accepte un DataFrame ou un Styler déjà formaté."""
    sty = df if hasattr(df, "set_table_styles") else df.style
    return (sty
              .set_caption(caption)
              .set_table_styles([
                  {"selector": "caption",
                   "props": [("caption-side", "top"), ("font-family", FONT),
                             ("font-size", "12px"), ("font-style", "italic"),
                             ("color", SLATE), ("padding-bottom", "6px")]},
                  {"selector": "th",
                   "props": [("background-color", DEEP), ("color", "white"),
                             ("font-family", FONT), ("font-size", "11.5px"),
                             ("text-align", "left"), ("padding", "6px 9px")]},
                  {"selector": "td",
                   "props": [("font-family", FONT), ("font-size", "12px"),
                             ("padding", "5px 9px"), ("border-bottom", f"1px solid {SAGE}")]},
              ]))

def fmt(n, d=1):
    """Formatage lisible des nombres."""
    if n is None or (isinstance(n, float) and not np.isfinite(n)): return "–"
    a = abs(n)
    if a >= 1e9:  return f"{n/1e9:.{d}f} bn"
    if a >= 1e6:  return f"{n/1e6:.{d}f} M"
    if a >= 1e3:  return f"{n/1e3:.{d}f} k"
    return f"{n:.{d}f}"

banner("INSTALLATION TERMINÉE", f"Environnement prêt — {ENV}",
       "Thème inspiré de la charte BAD chargé. Chaque graphique, carte et element du "
       "tableau de bord ci-dessous utilise la même palette et la même typographie.")

# 3 · Comprendre les deux sources de données

Avant d'écrire la moindre ligne d'analyse, un statisticien public doit pouvoir répondre à trois
questions sur toute source non traditionnelle : **quelle est l'unité d'observation, comment a-t-elle
été produite, et qu'ai-je le droit d'en publier ?**

## 3.1 Les données ouvertes Ookla Speedtest

| Propriété | Valeur |
|---|---|
| **Producteur** | Ookla® (Speedtest.net) |
| **Unité d'observation** | Un **carreau** : un carré au zoom 16 en projection Web-Mercator, soit ≈ **611 m × 611 m à l'équateur** |
| **Granularité temporelle** | **Trimestrielle**, du **T1 2019** à aujourd'hui |
| **Deux produits** | `fixed` (tests depuis le Wi-Fi ou le haut débit fixe) et `mobile` (tests depuis une connexion cellulaire) |
| **Champs clés** | `quadkey`, `avg_d_kbps`, `avg_u_kbps`, `avg_lat_ms`, `tests`, `devices` |
| **Règle d'agrégation** | Un carreau n'apparaît que s'il a reçu **≥ 1 test** dans le trimestre ; les valeurs sont des moyennes sur tous les tests du carreau |
| **Licence** | **CC BY-NC-SA 4.0** |
| **Diffusion** | Compartiment AWS S3 public `ookla-open-data`, en GeoParquet et Shapefile |

### Ce que la licence CC BY-NC-SA 4.0 implique pour un INS

C'est le point le plus important de cette séance, et celui que l'on saute le plus souvent.

- **BY** — vous devez créditer Ookla explicitement, dans le tableau de bord et dans toute publication.
- **NC** — **usage non commercial uniquement**. Vendre l'indicateur dérivé, ou l'intégrer à un produit payant, n'est pas autorisé. Une publication sur le site d'un INS comme bien public gratuit est normalement admissible ; *faites-le confirmer par écrit par votre service juridique avant la première diffusion*.
- **SA** — **partage dans les mêmes conditions** : toute œuvre dérivée que vous diffusez doit porter la même licence. Cela inclut votre tableau de bord et votre CSV agrégé. Cela signifie aussi que vous **ne pouvez pas** verser cet indicateur dans un portail de données ouvertes qui publie tout en CC BY ou CC0 sans signaler l'exception.
- Pour qu'un indicateur entre dans le système de production **officiel**, et non qu'il reste une diffusion expérimentale, la plupart des bureaux devront passer un accord direct avec Ookla ou avec les opérateurs nationaux.

### Le biais qu'il ne faut jamais oublier

Les données Ookla sont **participatives et auto-sélectionnées**. Un utilisateur lance un test parce
qu'il *soupçonne un problème*, ou parce qu'il *vient de souscrire une nouvelle connexion*.
Conséquences :

- Les carreaux sans données ne sont **pas** des carreaux sans couverture — ce sont des carreaux **sans test**. L'absence de mesure n'est pas l'absence de service.
- Les zones rurales et à faibles revenus sont systématiquement **sous-échantillonnées** : une moyenne nationale naïve est donc biaisée **vers le haut**.
- Les terminaux diffèrent (un téléphone ancien plafonne le débit mesuré), et un carreau avec 3 tests n'est pas comparable à un carreau avec 3 000.

C'est exactement pour cette raison que nous introduisons les données de population.

## 3.2 WorldPop

| Propriété | Valeur |
|---|---|
| **Producteur** | WorldPop, université de Southampton |
| **Produit utilisé** | Grille mondiale de population, ajustée aux estimations des Nations unies |
| **Résolution** | 1 km (par défaut ici) ou 100 m |
| **Unité** | Nombre estimé d'**habitants par cellule** |
| **Licence** | **CC BY 4.0** — plus permissive que celle d'Ookla |
| **Méthode** | Effectifs censitaires ou projetés redistribués par une forêt aléatoire dasymétrique utilisant le bâti, les routes, les lumières nocturnes et l'occupation du sol |

WorldPop permet de passer de *« le carreau moyen de ce district enregistre 18 Mbit/s »* — un énoncé
sur des **carrés** — à *« l'habitant médian de ce district vit là où 18 Mbit/s sont mesurés »* — un
énoncé sur des **personnes**. Seul le second est une statistique.

> **Avertissement :** WorldPop est lui-même un produit *modélisé*, construit en partie à partir des
> lumières nocturnes. Ne validez jamais un indicateur dérivé des lumières nocturnes contre WorldPop
> en le présentant comme une confirmation indépendante — la circularité est réelle.

## 3.3 Les quadkeys — la géométrie cachée dans une chaîne de caractères

Ookla livre la géométrie en WKT, mais il livre aussi le `quadkey`, et le quadkey **est** la
géométrie. Apprendre à le décoder vaut cinq minutes, parce que c'est ce qui rend possible la requête
distante de la section 5.

Un quadkey est le chemin de descente dans le quadtree Web-Mercator. À chaque niveau, le carré du
monde est divisé en quatre, numérotés ainsi :

```
      +---+---+
      | 0 | 1 |
      +---+---+
      | 2 | 3 |
      +---+---+
```

Ainsi la chaîne `"0313102310"` signifie : *prends le quadrant 0, puis le 3 de celui-ci, puis le 1 de
celui-là…* Chaque caractère ajoute un niveau de zoom : un **carreau au zoom 16 a donc un quadkey de
16 caractères**.

Trois propriétés nous importent :

1. **Préfixe = inclusion.** Tout carreau situé à l'intérieur du carreau de zoom 8 `03131023` commence par `03131023`. Une simple comparaison de chaînes remplace un index spatial.
2. **L'ordre lexicographique ≈ l'ordre spatial.** Trier les quadkeys regroupe les voisins — c'est pourquoi un filtre d'intervalle sur un fichier parquet trié élimine presque tout.
3. **Les chiffres entrelacent les bits de (x, y).** Le chiffre `d` contribue le bit `d & 1` à `x` et le bit `d >> 1` à `y`. Cela nous donne un décodeur exact et vectorisé en quatre lignes de NumPy.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Quadkey ⇄ géographie — sans bibliothèque externe, entièrement vectorisé
# ══════════════════════════════════════════════════════════════════════════════
R_EARTH_M = 6378137.0

def quadkeys_to_xy(quadkeys, zoom=16):
    """Quadkey vectorisé -> coordonnées entières (tile_x, tile_y) de la grille.

    Chaque caractère du quadkey est un chiffre en base 4 dont les bits sont entrelacés :
    le bit 0 appartient a X, le bit 1 appartient a Y.
    """
    qk = np.asarray(quadkeys, dtype=f"U{zoom}")
    # on lit le tableau unicode comme des points de code -> forme (n, zoom)
    codes = qk.view(np.uint32).reshape(-1, zoom) - ord("0")
    shifts = (zoom - 1 - np.arange(zoom)).astype(np.uint32)
    x = ((codes & 1) << shifts).sum(axis=1)
    y = ((codes >> 1) << shifts).sum(axis=1)
    return x.astype(np.int64), y.astype(np.int64)


def xy_to_lonlat(x, y, zoom=16):
    """Coordonnées de grille (éventuellement fractionnaires) -> lon/lat en EPSG:4326."""
    n = 2.0 ** zoom
    lon = x / n * 360.0 - 180.0
    lat = np.degrees(np.arctan(np.sinh(np.pi * (1.0 - 2.0 * y / n))))
    return lon, lat


def quadkeys_to_bounds(quadkeys, zoom=16):
    """-> DataFrame avec le centroïde et la boîte englobante du carreau, en degrés."""
    x, y = quadkeys_to_xy(quadkeys, zoom)
    west,  north = xy_to_lonlat(x,     y,     zoom)
    east,  south = xy_to_lonlat(x + 1, y + 1, zoom)
    clon,  clat  = xy_to_lonlat(x + .5, y + .5, zoom)
    return pd.DataFrame({"tile_x": x, "tile_y": y,
                         "west": west, "south": south, "east": east, "north": north,
                         "lon": clon, "lat": clat})


def lonlat_to_quadkey(lon, lat, zoom=16):
    """lon/lat scalaires -> chaîne quadkey. Sert à construire la requête distante."""
    lat = max(min(lat, 85.05112878), -85.05112878)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    s = math.sin(math.radians(lat))
    y = int((0.5 - math.log((1 + s) / (1 - s)) / (4 * math.pi)) * n)
    x, y = min(max(x, 0), n - 1), min(max(y, 0), n - 1)
    out = []
    for i in range(zoom, 0, -1):
        digit, mask = 0, 1 << (i - 1)
        if x & mask: digit += 1
        if y & mask: digit += 2
        out.append(str(digit))
    return "".join(out)


def tile_area_km2(lat, zoom=16):
    """Surface au sol reelle d'un carreau Web-Mercator a une latitude donnee."""
    side_m = (2 * np.pi * R_EARTH_M / (2 ** zoom)) * np.cos(np.radians(lat))
    return (side_m ** 2) / 1e6


# --- vérification : Kigali, Rwanda -------------------------------------------
_qk = lonlat_to_quadkey(30.0619, -1.9441, 16)
_b  = quadkeys_to_bounds([_qk]).iloc[0]
print(f"Kigali (30.0619, -1.9441)")
print(f"  quadkey z16   : {_qk}")
print(f"  carreau x / y : {int(_b.tile_x)} / {int(_b.tile_y)}")
print(f"  centroide     : {_b.lon:.5f}, {_b.lat:.5f}")
print(f"  côté du carreau : {(2*np.pi*R_EARTH_M/2**16)*np.cos(np.radians(_b.lat)):.0f} m")
print(f"  surface       : {tile_area_km2(_b.lat):.4f} km²")
assert lonlat_to_quadkey(_b.lon, _b.lat, 16) == _qk, "aller-retour incorrect"
print("  aller-retour  : OK ✓")

# 4 · Limites administratives

Nous utilisons **geoBoundaries** (diffusion `gbOpen`, CC BY 4.0, W. M. Geolab / William & Mary)
parce qu'elle est ouverte, mondiale, versionnée et citable. Deux réserves à mentionner dans votre
propre README :

- geoBoundaries n'est **pas** votre référentiel national faisant autorité. Si votre INS ou votre
  institut cartographique national publie des limites officielles, utilisez-les : il suffit de
  remplacer la fonction ci-dessous, tout le reste continue de fonctionner.
- Les versions des limites évoluent. Consignez la version utilisée ; l'API l'expose.

Nous conservons aussi le polygone **ADM0** (national), qui servira à découper les carreaux Ookla.

In [ ]:
GB_API = "https://www.geoboundaries.org/api/current/gbOpen/{iso}/{lvl}/"
GB_RAW = ("https://raw.githubusercontent.com/wmgeolab/geoBoundaries/main/releaseData/"
          "gbOpen/{iso}/{lvl}/geoBoundaries-{iso}-{lvl}_simplified.geojson")


def get_boundaries(iso3, level):
    """Telecharge une couche geoBoundaries en GeoDataFrame (API d'abord, GitHub brut en secours)."""
    cache = Path(CACHE_DIR) / f"bnd_{iso3}_{level}.geojson"
    if cache.exists():
        return gpd.read_file(cache)

    url = None
    try:
        meta = requests.get(GB_API.format(iso=iso3, lvl=level), timeout=45).json()
        if isinstance(meta, list):
            meta = meta[0]
        url = meta.get("gjDownloadURL") or meta.get("simplifiedGeometryGeoJSON")
    except Exception:
        pass
    if not url:
        url = GB_RAW.format(iso=iso3, lvl=level)

    gdf = gpd.read_file(url)
    gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    gdf.to_file(cache, driver="GeoJSON")
    return gdf


# --- contour national --------------------------------------------------------
adm0 = get_boundaries(COUNTRY_ISO3, "ADM0")
COUNTRY_NAME = str(adm0.iloc[0].get("shapeGroup", COUNTRY_ISO3))
for c in ("shapeName", "shapeGroupName"):
    if c in adm0.columns and isinstance(adm0.iloc[0][c], str):
        COUNTRY_NAME = adm0.iloc[0][c]
        break
COUNTRY_GEOM = adm0.union_all() if hasattr(adm0, "union_all") else adm0.unary_union
BBOX = COUNTRY_GEOM.bounds                      # (west, south, east, north)

# --- unités infranationales -------------------------------------------------------
try:
    admin = get_boundaries(COUNTRY_ISO3, ADMIN_LEVEL)
    if admin.empty:
        raise ValueError("empty layer")
except Exception as e:
    print(f"{ADMIN_LEVEL} indisponible ({e}) — repli sur ADM1")
    ADMIN_LEVEL = "ADM1"
    admin = get_boundaries(COUNTRY_ISO3, "ADM1")

admin = admin.rename(columns={"shapeName": "admin_name", "shapeID": "admin_id"})
if "admin_name" not in admin.columns:
    admin["admin_name"] = [f"{ADMIN_LEVEL}-{i+1}" for i in range(len(admin))]
admin["admin_name"] = admin["admin_name"].astype(str)
admin = admin[["admin_name", "geometry"]].reset_index(drop=True)

# les noms administratifs ne sont pas garantis uniques -> lever l'ambiguïté avant toute jointure
dupes = admin["admin_name"].duplicated(keep=False)
if dupes.any():
    admin.loc[dupes, "admin_name"] = (admin.loc[dupes, "admin_name"] + " ("
                                      + (admin.loc[dupes].groupby("admin_name").cumcount() + 1).astype(str)
                                      + ")")
    print(f"{int(dupes.sum())} noms {ADMIN_LEVEL} en double ont été désambiguïsés.")
admin["admin_idx"] = admin.index

# un second niveau, plus grossier, pour les croisements quand on travaille en ADM2
if ADMIN_LEVEL == "ADM2":
    try:
        adm1 = get_boundaries(COUNTRY_ISO3, "ADM1").rename(columns={"shapeName": "region"})
        adm1 = adm1[["region", "geometry"]]
    except Exception:
        adm1 = None
else:
    adm1 = None

# --- surface en km2 (projection équivalente ; ne jamais calculer une aire en degrés) ------
admin["area_km2"] = admin.to_crs("EPSG:6933").area / 1e6
COUNTRY_AREA_KM2 = float(gpd.GeoSeries([COUNTRY_GEOM], crs=4326)
                         .to_crs("EPSG:6933").area.iloc[0] / 1e6)

banner("LIMITES CHARGÉES", f"{COUNTRY_NAME} ({COUNTRY_ISO3})",
       f"{len(admin)} unités {ADMIN_LEVEL} &nbsp;·&nbsp; "
       f"{COUNTRY_AREA_KM2:,.0f} km² &nbsp;·&nbsp; "
       f"bbox {BBOX[0]:.2f}, {BBOX[1]:.2f} → {BBOX[2]:.2f}, {BBOX[3]:.2f} &nbsp;·&nbsp; "
       f"source : geoBoundaries gbOpen")

display(style_table(admin[["admin_name", "area_km2"]]
                    .sort_values("area_km2", ascending=False).head(8)
                    .style.format({"area_km2": "{:,.0f}"}),
                    f"Plus grandes unités {ADMIN_LEVEL} (km²) — les 8 premières sur {len(admin)}"))

# 5 · Acquérir les carreaux Ookla — sans télécharger la planète

## 5.1 Où vivent les données

Ookla publie un fichier parquet par trimestre et par service, sur un compartiment S3 public :

```
https://ookla-open-data.s3.amazonaws.com/parquet/performance/
    type={fixed|mobile}/year=AAAA/quarter=T/AAAA-MM-01_performance_{type}_tiles.parquet
```

Chaque fichier est **mondial** et pèse de **200 Mo à 1 Go**. Le Rwanda représente environ **0,02 %**
des lignes. Télécharger le fichier entier pour en conserver 0,02 % est l'erreur la plus courante de
ce laboratoire.

## 5.2 La technique : requêtes HTTP par plages + élagage par intervalles de quadkeys

Parquet est un format **en colonnes** organisé en *groupes de lignes*, et chaque groupe de lignes
stocke dans son pied de fichier les **valeurs minimale et maximale de chaque colonne**. Un moteur de
requête capable de lire ce pied de fichier par HTTP peut donc :

1. lire le pied de fichier (quelques kilooctets),
2. écarter tout groupe de lignes dont l'intervalle `[min(quadkey), max(quadkey)]` ne croise pas les
   intervalles de quadkeys de notre pays,
3. récupérer, par des requêtes HTTP sur plages d'octets, **uniquement les plages survivantes**, et
4. décompresser **uniquement les colonnes demandées**.

Comme les fichiers sont triés par quadkey et que les quadkeys sont spatialement cohérents
(section 3.3), un pays ne survit généralement que dans une poignée de groupes de lignes. En
pratique : **quelques dizaines de mégaoctets transférés au lieu de plusieurs centaines**, en
quelques secondes.

Nous utilisons **DuckDB** avec son extension `httpfs`, et nous retombons sur `pyarrow.dataset` si
l'extension ne peut pas être installée — ce qui arrive dans certains environnements d'entreprise
verrouillés.

## 5.3 Construire le prédicat

Nous recouvrons la boîte englobante du pays avec des carreaux à **faible zoom** (typiquement z = 6
à 9), convertissons chacun en son préfixe de quadkey, et transformons chaque préfixe `p` en un
intervalle de chaînes fermé :

```
p + "0000…"   ≤  quadkey  ≤   p + "3333…"
```

Nous ajoutons également une clause **globale** `BETWEEN min … max` : un prédicat d'intervalle unique
est la forme que le moteur pousse le plus sûrement jusqu'aux statistiques des groupes de lignes. Les
clauses par préfixe éliminent ensuite les carreaux qui tombent dans la boîte englobante mais hors du
pays.

In [ ]:
OOKLA_BASE = "https://ookla-open-data.s3.amazonaws.com/parquet/performance"
Q_MONTH = {1: "01", 2: "04", 3: "07", 4: "10"}


def ookla_url(service, year, quarter):
    return (f"{OOKLA_BASE}/type={service}/year={year}/quarter={quarter}/"
            f"{year}-{Q_MONTH[quarter]}-01_performance_{service}_tiles.parquet")


def quarter_exists(service, year, quarter, timeout=25):
    try:
        r = requests.head(ookla_url(service, year, quarter), timeout=timeout)
        return r.status_code == 200
    except Exception:
        return False


def previous_quarter(year, quarter):
    return (year - 1, 4) if quarter == 1 else (year, quarter - 1)


def recent_quarters(n, services=("mobile",), max_back=10):
    """Renvoie les n trimestres publiés les plus récents, du plus récent au plus ancien."""
    today = dt.date.today()
    y, q = today.year, (today.month - 1) // 3 + 1
    found, tried = [], 0
    while len(found) < n and tried < max_back:
        if all(quarter_exists(s, y, q) for s in services):
            found.append((y, q))
        elif not found:
            pass                      # on remonte encore jusqu'au dernier trimestre publié
        y, q = previous_quarter(y, q)
        tried += 1
    return found


QUARTERS = recent_quarters(N_QUARTERS, services=SERVICES)
if not QUARTERS:
    raise RuntimeError("Aucun trimestre Ookla publié n'a été trouvé — vérifiez votre connexion.")

LATEST_Y, LATEST_Q = QUARTERS[0]
banner("CATALOGUE OOKLA", f"Dernier trimestre publié : {LATEST_Y} T{LATEST_Q}",
       "Trimestres à récupérer : " +
       " · ".join(f"{y} Q{q}" for y, q in QUARTERS) +
       f"<br>Exemple de fichier : <code style='font-size:11px'>{ookla_url(SERVICES[0], LATEST_Y, LATEST_Q)}</code>")

In [ ]:
def bbox_quadkey_prefixes(bbox, max_prefixes=160):
    """Recouvre une boîte lon/lat de préfixes de quadkeys, au zoom le plus profond
    qui garde un nombre de préfixes raisonnable (plus profond = élagage plus sélectif)."""
    west, south, east, north = bbox
    for zoom in range(10, 2, -1):
        n = 2 ** zoom
        x0 = int((west + 180) / 360 * n); x1 = int((east + 180) / 360 * n)
        def _y(lat):
            s = math.sin(math.radians(max(min(lat, 85.05), -85.05)))
            return int((0.5 - math.log((1 + s) / (1 - s)) / (4 * math.pi)) * n)
        y0, y1 = _y(north), _y(south)
        count = (x1 - x0 + 1) * (y1 - y0 + 1)
        if count <= max_prefixes:
            prefixes = []
            for xx in range(x0, x1 + 1):
                for yy in range(y0, y1 + 1):
                    lon = (xx + 0.5) / n * 360 - 180
                    lat = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (yy + 0.5) / n))))
                    prefixes.append(lonlat_to_quadkey(lon, lat, zoom))
            return sorted(set(prefixes)), zoom
    raise RuntimeError("Impossible de construire les préfixes de quadkeys pour cette boîte.")


PREFIXES, PREFIX_ZOOM = bbox_quadkey_prefixes(BBOX)
PAD = 16 - PREFIX_ZOOM
RANGES = [(p + "0" * PAD, p + "3" * PAD) for p in PREFIXES]
GLOBAL_LO, GLOBAL_HI = RANGES[0][0], RANGES[-1][1]

print(f"Boîte englobante   : {BBOX[0]:.3f}, {BBOX[1]:.3f} → {BBOX[2]:.3f}, {BBOX[3]:.3f}")
print(f"Zoom des préfixes  : z{PREFIX_ZOOM}  ({len(PREFIXES)} prefixes)")
print(f"Intervalle global  : {GLOBAL_LO}  →  {GLOBAL_HI}")
print(f"Premiers préfixes  : {', '.join(PREFIXES[:6])}{' …' if len(PREFIXES) > 6 else ''}")
print(f"\nSelectivite        : l'intervalle global couvre "
      f"{100 * (len(PREFIXES) / 4 ** PREFIX_ZOOM):.4f} % des carreaux z{PREFIX_ZOOM} du monde")

In [ ]:
OOKLA_COLS = ["quadkey", "avg_d_kbps", "avg_u_kbps", "avg_lat_ms", "tests", "devices"]

_DUCK = None
def duck():
    """Crée paresseusement une connexion DuckDB avec httpfs activé."""
    global _DUCK
    if _DUCK is None:
        con = duckdb.connect()
        try:
            con.execute("INSTALL httpfs; LOAD httpfs;")
        except Exception as e:
            print(f"  httpfs indisponible ({e}) — repli sur pyarrow")
            raise
        con.execute("SET enable_progress_bar=false;")
        _DUCK = con
    return _DUCK


def available_columns(url):
    """Ne lit que le pied du parquet pour connaître les colonnes réellement présentes."""
    d = duck().execute(f"DESCRIBE SELECT * FROM read_parquet('{url}') LIMIT 0").df()
    return list(d["column_name"])


def _fetch_duckdb(url):
    cols = [c for c in OOKLA_COLS if c in available_columns(url)] or OOKLA_COLS
    where = " OR ".join(f"(quadkey >= '{lo}' AND quadkey <= '{hi}')" for lo, hi in RANGES)
    sql = f"""
        SELECT {', '.join(cols)}
        FROM read_parquet('{url}')
        WHERE quadkey >= '{GLOBAL_LO}' AND quadkey <= '{GLOBAL_HI}'
          AND ({where})
    """
    return duck().execute(sql).df()


def _fetch_pyarrow(url):
    """Solution de repli : dataset pyarrow sur HTTP, avec le meme predicat d'intervalle."""
    import pyarrow.dataset as ds, pyarrow.compute as pc, fsspec
    fs, path = fsspec.core.url_to_fs(url)
    dataset = ds.dataset(path, filesystem=fs, format="parquet")
    f = None
    for lo, hi in RANGES:
        cond = (pc.field("quadkey") >= lo) & (pc.field("quadkey") <= hi)
        f = cond if f is None else (f | cond)
    return dataset.to_table(columns=OOKLA_COLS, filter=f).to_pandas()


def fetch_ookla(service, year, quarter, verbose=True):
    """Sous-ensemble national d'un trimestre Ookla — mis en cache local en parquet."""
    cache = Path(CACHE_DIR) / f"ookla_{COUNTRY_ISO3}_{service}_{year}Q{quarter}.parquet"
    if cache.exists():
        df = pd.read_parquet(cache)
        if verbose:
            print(f"  {service:<6} {year} T{quarter}  ·  {len(df):>7,} carreaux  (cache)")
        return df

    url = ookla_url(service, year, quarter)
    t0 = time.time()
    try:
        df = _fetch_duckdb(url)
    except Exception as e:
        if verbose:
            print(f"  Échec de la voie DuckDB ({type(e).__name__}) — essai avec pyarrow …")
        try:
            df = _fetch_pyarrow(url)
        except Exception as e2:
            raise RuntimeError(
                f"Lecture impossible : {url}\nDuckDB : {e}\npyarrow : {e2}\n"
                "Sous Kaggle, vérifiez qu'Internet est ACTIVÉ dans le panneau de réglages."
            ) from e2

    df["service"], df["year"], df["quarter"] = service, year, quarter
    df.to_parquet(cache, index=False)
    if verbose:
        print(f"  {service:<6} {year} T{quarter}  ·  {len(df):>7,} carreaux  "
              f"({time.time() - t0:5.1f}s à distance)")
    return df


print(f"Récupération de {len(SERVICES)} service(s) × {len(QUARTERS)} trimestre(s) "
      f"depuis les fichiers parquet mondiaux …\n")
frames = []
for (y, q) in QUARTERS:
    for s in SERVICES:
        try:
            frames.append(fetch_ookla(s, y, q))
        except Exception as e:
            print(f"  ! {s} {y}T{q} ignoré : {e}")

raw = pd.concat(frames, ignore_index=True)
print(f"\nLignes récupérées au total (boîte englobante) : {len(raw):,}")

# 6 · Nettoyage, géocodage et découpage

Trois opérations, dans cet ordre :

1. **Unités.** Ookla stocke les débits en **kbit/s** ; tout indicateur publié doit être en
   **Mbit/s** (`Mbit/s = kbit/s / 1000`). Se tromper d'un facteur 1000 est l'erreur classique du
   premier lancement.
2. **Géométrie.** Nous décodons le quadkey avec la fonction vectorisée de la section 3.3 — sans
   analyser le WKT, ce qui serait un ordre de grandeur plus lent sur les grands pays.
3. **Découpage.** La boîte englobante est un rectangle ; le pays n'en est pas un. Nous conservons
   les carreaux dont le **centroïde** tombe à l'intérieur du polygone ADM0. Le centroïde dans le
   polygone est la convention standard et reproductible pour une grille de 611 m ; énoncez-la dans
   votre méthodologie.

In [ ]:
df = raw.copy()

# 1 · unités et champs dérivés ------------------------------------------------
df["d_mbps"] = df["avg_d_kbps"] / 1000.0
df["u_mbps"] = df["avg_u_kbps"] / 1000.0
df["latency_ms"] = df["avg_lat_ms"] if "avg_lat_ms" in df.columns else np.nan
df["tests_per_device"] = df["tests"] / df["devices"].replace(0, np.nan)

# 2 · géométrie déduite du quadkey ----------------------------------------------
bounds = quadkeys_to_bounds(df["quadkey"].values, zoom=16)
df = pd.concat([df.reset_index(drop=True), bounds], axis=1)
df["tile_km2"] = tile_area_km2(df["lat"].values)

# 3 · découpage sur le polygone national (règle du centroïde) ---------------------------
pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")
inside = pts.within(COUNTRY_GEOM)
n_before = len(df)
df = df[inside.values].reset_index(drop=True)

# 4 · filtre de qualité ----------------------------------------------------------
df = df[df["tests"] >= MIN_TESTS_TILE]
df = df[(df["d_mbps"] > 0) & (df["d_mbps"] < 5000)].reset_index(drop=True)

banner("CARREAUX NETTOYÉS", f"{len(df):,} carreaux conservés dans {COUNTRY_NAME}",
       f"{n_before - len(df):,} lignes écartées : hors du polygone national, "
       f"moins de {MIN_TESTS_TILE} test(s), ou hors des bornes physiques. "
       f"Débits convertis de kbit/s en Mbit/s.")

display(style_table(
    df[["quadkey", "service", "year", "quarter", "lon", "lat",
        "d_mbps", "u_mbps", "latency_ms", "tests", "devices"]].head(6)
      .style.format({"lon": "{:.4f}", "lat": "{:.4f}", "d_mbps": "{:.1f}",
                     "u_mbps": "{:.1f}", "latency_ms": "{:.0f}"}),
    "Carreaux Ookla nettoyés — 6 premières lignes"))

# 7 · Diagnostic de couverture — *avant* de concevoir le moindre indicateur

> **Vérifiez la couverture nationale avant de concevoir tout indicateur.**

C'est l'étape qui sépare une statistique expérimentale d'une statistique trompeuse. La question
n'est pas « quel est le débit moyen ? » mais **« sur quelle part du pays, et sur combien de
personnes, ai-je effectivement une mesure ? »**

Nous y répondons sur trois axes :

- **Spatial** : part du territoire national couverte par au moins un carreau mesuré.
- **Volume** : combien de tests, et à quel point ils sont concentrés (quelques carreaux peuvent
  porter la majorité des tests).
- **Stabilité** : le nombre de carreaux mesurés est-il stable d'un trimestre à l'autre, ou
  s'effondre-t-il ?

Si la couverture spatiale est de quelques pour cent, une moyenne *nationale* n'est pas défendable —
mais un indicateur *infranational, pondéré par la population, restreint à l'urbain* l'est souvent
encore. Dites lequel vous publiez.

In [ ]:
latest = df[(df.year == LATEST_Y) & (df.quarter == LATEST_Q)]

diag = []
for s in SERVICES:
    d = latest[latest.service == s]
    if d.empty:
        continue
    covered_km2 = d["tile_km2"].sum()
    # concentration : part des tests portée par le 1 % de carreaux les plus actifs
    t = np.sort(d["tests"].values)[::-1]
    top1 = t[:max(1, len(t) // 100)].sum() / t.sum() * 100 if t.sum() else np.nan
    diag.append({
        "Service": s,
        "Carreaux mesurés": len(d),
        "Surface couverte (km²)": covered_km2,
        "Territoire couvert (%)": 100 * covered_km2 / COUNTRY_AREA_KM2,
        "Tests": int(d["tests"].sum()),
        "Terminaux": int(d["devices"].sum()),
        "Tests dans le 1 % de carreaux les plus actifs (%)": top1,
        "Tests médians / carreau": float(d["tests"].median()),
    })
diag = pd.DataFrame(diag)

display(style_table(diag.style.format({
    "Carreaux mesurés": "{:,.0f}", "Surface couverte (km²)": "{:,.0f}",
    "Territoire couvert (%)": "{:.2f}", "Tests": "{:,.0f}", "Terminaux": "{:,.0f}",
    "Tests dans le 1 % de carreaux les plus actifs (%)": "{:.1f}", "Tests médians / carreau": "{:.0f}"}),
    f"Diagnostic de couverture — {COUNTRY_NAME}, {LATEST_Y} T{LATEST_Q}"))

_land = diag["Territoire couvert (%)"].max() if not diag.empty else 0
if _land < 2:
    callout(f"<b>La couverture spatiale est très faible ({_land:.2f} % du territoire).</b> "
            "Une moyenne nationale serait un énoncé sur une poignée de carrés urbains. "
            "Publiez un indicateur urbain ou pondéré par la population avec un dénominateur "
            "explicite, ou traitez le résultat comme un simple outil de diagnostic.", "risk")
elif _land < 10:
    callout(f"<b>La couverture spatiale atteint {_land:.2f} % du territoire</b> — normal pour cette "
            "source. La pondération par la population (section 9) est obligatoire avant de citer "
            "le moindre chiffre.", "warn")
else:
    callout(f"La couverture spatiale atteint {_land:.2f} % du territoire — comparativement élevée. "
            "La pondération par la population reste nécessaire, mais les ventilations "
            "infranationales seront robustes.", "info")

In [ ]:
# --- stabilité d'un trimestre à l'autre ----------------------------------------------
if len(QUARTERS) > 1:
    stab = (df.groupby(["year", "quarter", "service"])
              .agg(tiles=("quadkey", "size"), tests=("tests", "sum"),
                   median_d=("d_mbps", "median"))
              .reset_index())
    stab["period"] = stab["year"].astype(str) + " Q" + stab["quarter"].astype(str)
    stab = stab.sort_values(["year", "quarter"])
    display(style_table(
        stab.pivot(index="period", columns="service", values="tiles")
            .fillna(0).astype(int).reset_index()
            .style.format(thousands=","),
        "Carreaux mesurés par trimestre — une chute brutale signale en général un problème de données, pas un changement de réseau"))
else:
    stab = None
    print("Un seul trimestre sélectionné — mettez N_QUARTERS > 1 pour activer le contrôle de stabilité.")

# 8 · WorldPop — faire entrer les habitants dans le tableau

Nous téléchargeons la **grille mondiale de population ajustée aux estimations des Nations unies**
pour le pays. Deux produits sont câblés :

| Réglage | Produit | Taille typique | Usage |
|---|---|---|---|
| `"1km"` | `Global_2000_2020_1km_UNadj` | 0,1 à 20 Mo | **Par défaut.** Rapide, suffisant aux niveaux ADM1/ADM2 |
| `"100m"` | `Global_2000_2020` (non contraint) | 20 à 600 Mo | Précision au carreau, lent pour les grands pays |

La version ajustée aux Nations unies remet la grille à l'échelle pour que le **total national
corresponde à l'estimation des Perspectives démographiques mondiales** — c'est ce qui rend le
résultat comparable au reste de votre système statistique.

In [ ]:
WP_1KM  = ("https://data.worldpop.org/GIS/Population/Global_2000_2020_1km_UNadj/"
           "{year}/{ISO}/{iso}_ppp_{year}_1km_Aggregated_UNadj.tif")
WP_100M = ("https://data.worldpop.org/GIS/Population/Global_2000_2020/"
           "{year}/{ISO}/{iso}_ppp_{year}.tif")
WP_REST = "https://www.worldpop.org/rest/data/pop/wpgp?iso3={ISO}"


def download(url, dest, label=""):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 1024:
        print(f"  {label or dest.name} : en cache ({dest.stat().st_size/1e6:.1f} Mo)")
        return dest
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done, t0 = 0, time.time()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk); done += len(chunk)
                if total:
                    pct = 100 * done / total
                    print(f"\r  {label} {pct:5.1f} %  ({done/1e6:.1f}/{total/1e6:.1f} Mo)",
                          end="", flush=True)
    print(f"\r  {label} : {done/1e6:.1f} Mo en {time.time()-t0:.1f}s" + " " * 20)
    return dest


def worldpop_raster(iso3, year, res):
    tpl = WP_1KM if res == "1km" else WP_100M
    url = tpl.format(year=year, ISO=iso3.upper(), iso=iso3.lower())
    dest = Path(CACHE_DIR) / f"worldpop_{iso3}_{year}_{res}.tif"
    try:
        return download(url, dest, f"WorldPop {iso3} {year} {res}")
    except Exception as e:
        print(f"  URL directe en échec ({e}) — interrogation du catalogue REST WorldPop …")
        meta = requests.get(WP_REST.format(ISO=iso3.upper()), timeout=60).json()
        files = [f for rec in meta.get("data", []) if str(rec.get("popyear")) == str(year)
                 for f in rec.get("files", []) if str(f).endswith(".tif")]
        if not files:
            raise RuntimeError(f"Aucun raster WorldPop trouvé pour {iso3} {year}")
        return download(files[0], dest, f"WorldPop {iso3} (REST)")


POP_TIF = worldpop_raster(COUNTRY_ISO3, WORLDPOP_YEAR, WORLDPOP_RES)

with rasterio.open(POP_TIF) as src:
    print(f"\n  SRC         : {src.crs}")
    print(f"  Taille      : {src.width} × {src.height} px")
    print(f"  Taille pixel: {abs(src.transform.a):.6f}° ≈ "
          f"{abs(src.transform.a) * 111.32:.2f} km à l'équateur")
    print(f"  Sans-donnée : {src.nodata}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Raster  ->  table ordonnée des cellules peuplées
# ══════════════════════════════════════════════════════════════════════════════
with rasterio.open(POP_TIF) as src:
    win = rasterio.windows.from_bounds(*BBOX, transform=src.transform)
    win = win.round_offsets().round_lengths()
    win = rasterio.windows.intersection(
        win, rasterio.windows.Window(0, 0, src.width, src.height))
    arr = src.read(1, window=win).astype("float64")
    wtr = src.window_transform(win)
    nod = src.nodata

valid = np.isfinite(arr) & (arr > -1e30)
if nod is not None:
    valid &= (arr != nod)
arr = np.where(valid, np.clip(arr, 0, None), np.nan)

rows, cols = np.nonzero(valid)
pop_vals = arr[rows, cols]

# centroïdes des cellules (centre du pixel) en EPSG:4326
clon = wtr.c + wtr.a * (cols + 0.5) + wtr.b * (rows + 0.5)
clat = wtr.f + wtr.d * (cols + 0.5) + wtr.e * (rows + 0.5)

# surface au sol réelle de chaque cellule
dlon, dlat = abs(wtr.a), abs(wtr.e)
cell_km2 = (dlat * 111.32) * (dlon * 111.32 * np.cos(np.radians(clat)))

cells = pd.DataFrame({
    "row": rows.astype(np.int64), "col": cols.astype(np.int64),
    "lon": clon, "lat": clat, "pop": pop_vals, "cell_km2": cell_km2,
})
cells["cell_id"] = cells["row"] * (arr.shape[1] + 1) + cells["col"]
cells["density"] = cells["pop"] / cells["cell_km2"]

# découpage sur le polygone national
cpts = gpd.GeoDataFrame(cells, geometry=gpd.points_from_xy(cells.lon, cells.lat), crs="EPSG:4326")
cells = cells[cpts.within(COUNTRY_GEOM).values].reset_index(drop=True)

POP_TOTAL = float(cells["pop"].sum())
kpi_row([
    ("Cellules peuplées", f"{len(cells):,}", "", DEEP),
    ("Population totale", fmt(POP_TOTAL), f"({WORLDPOP_YEAR})", GREEN),
    ("Densité moyenne", f"{POP_TOTAL / COUNTRY_AREA_KM2:,.0f}", "/km²", TEAL),
    ("Résolution de la grille", f"{dlon * 111.32:.2f}", "km", OCHRE),
])
callout("Confrontez ce total à votre propre projection nationale pour "
        f"{WORLDPOP_YEAR}. Un écart de plus de quelques pour cent mérite une phrase dans votre "
        "méthodologie — WorldPop est un modèle, pas un recensement.", "warn")

# 9 · Intégration spatiale — le cœur du laboratoire

Nous disposons maintenant de deux grilles qui **ne coïncident pas** :

```
   WorldPop  :  cellules de ~1 000 m,  couverture complète du territoire
   Ookla     :  carreaux de ~611 m,    uniquement là où quelqu'un a lancé un test
```

Plutôt que de feindre une précision que nous n'avons pas, nous appliquons une règle explicite et
défendable :

1. **Localiser chaque carreau Ookla dans la grille de population** par son centroïde → chaque
   carreau appartient à exactement une cellule WorldPop.
2. Une cellule est **mesurée** si elle contient au moins un carreau ; sinon, c'est une **lacune de
   mesure**. Nous obtenons ainsi le dénominateur qui compte : `population vivant dans une cellule
   mesurée / population totale`.
3. **Répartir** la population d'une cellule mesurée à parts égales entre les carreaux qu'elle
   contient. Chaque carreau reçoit un poids de population `pop_cellule / n_carreaux_dans_la_cellule`.
   Cumulée sur le pays, la population répartie égale exactement la population des cellules mesurées :
   les poids sont cohérents.

> **Pourquoi pas une véritable intersection pondérée par les surfaces ?** Parce qu'avec une grille
> de population de 1 km, cela ajouterait de la fausse précision, pas de l'exactitude. Si vous passez
> `WORLDPOP_RES` à `"100m"`, l'affectation inverse (cellule de population → carreau contenant)
> devient exacte et ce même code la traite sans modification.

L'opération ci-dessous est en NumPy pur — aucune jointure spatiale — et s'exécute en quelques
millisecondes, même pour le Nigeria.

In [ ]:
# --- 1 · chaque carreau -> sa cellule de population -----------------------------------
inv = ~wtr                                        # transformation affine inverse
tcol, trow = inv * (df["lon"].values, df["lat"].values)
r = np.floor(trow).astype(np.int64)
c = np.floor(tcol).astype(np.int64)
ok = (r >= 0) & (r < arr.shape[0]) & (c >= 0) & (c < arr.shape[1])
df["row"], df["col"] = r, c
df["cell_id"] = np.where(ok, r * (arr.shape[1] + 1) + c, -1)

# --- 2 · combien de carreaux se partagent chaque cellule (par service et trimestre) ------------
key = ["cell_id", "service", "year", "quarter"]
df["n_tiles_in_cell"] = df.groupby(key)["quadkey"].transform("size")

# --- 3 · rattacher la population de la cellule et la répartir ------------------------------
df = df.merge(cells[["cell_id", "pop", "cell_km2", "density"]], on="cell_id", how="left")
df["pop"] = df["pop"].fillna(0.0)
df["pop_tile"] = df["pop"] / df["n_tiles_in_cell"]

# --- 4 · classe d'urbanisation (approximation par la densité, inspirée de GHSL) -------------------
def urban_class(d):
    return np.where(d >= URBAN_DENS_MIN, "Urban",
           np.where(d >= PERIURBAN_DENS_MIN, "Peri-urban", "Rural"))

df["settlement"]    = urban_class(df["density"].fillna(0).values)
cells["settlement"] = urban_class(cells["density"].values)

# --- 5 · indicateur mesuré / lacune, par service, dernier trimestre --------------------
for s in SERVICES:
    measured = set(df.loc[(df.service == s) & (df.year == LATEST_Y) &
                          (df.quarter == LATEST_Q), "cell_id"])
    cells[f"measured_{s}"] = cells["cell_id"].isin(measured)
cells["measured_any"] = cells[[f"measured_{s}" for s in SERVICES]].any(axis=1)

alloc = df.loc[(df.year == LATEST_Y) & (df.quarter == LATEST_Q)].groupby("service")["pop_tile"].sum()
print("Population répartie sur les carreaux mesurés (dernier trimestre)")
for s in SERVICES:
    cov = cells.loc[cells[f"measured_{s}"], "pop"].sum()
    print(f"  {s:<6}: {fmt(alloc.get(s, 0)):>8} répartis  ·  "
          f"{fmt(cov):>8} vivant dans une cellule mesurée  ·  "
          f"{100*cov/POP_TOTAL:5.1f} % de la population")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Rattacher les unités administratives (une jointure spatiale pour les cellules, une pour les carreaux)
# ══════════════════════════════════════════════════════════════════════════════
def attach_admin(frame, admin_gdf, label="admin_name"):
    g = gpd.GeoDataFrame(frame[["lon", "lat"]].copy(),
                         geometry=gpd.points_from_xy(frame["lon"], frame["lat"]),
                         crs="EPSG:4326")
    j = gpd.sjoin(g, admin_gdf[[label, "geometry"]], how="left", predicate="within")
    j = j[~j.index.duplicated(keep="first")]
    return j[label].reindex(frame.index).values

t0 = time.time()
cells["admin_name"] = attach_admin(cells, admin)
df["admin_name"]    = attach_admin(df, admin)
if adm1 is not None:
    cells["region"] = attach_admin(cells, adm1, "region")
    df["region"]    = attach_admin(df, adm1, "region")
print(f"Jointures spatiales effectuées en {time.time()-t0:.1f}s "
      f"({len(cells):,} cellules + {len(df):,} carreaux)")

unmatched = df["admin_name"].isna().mean() * 100
if unmatched > 1:
    callout(f"{unmatched:.1f} % des carreaux tombent hors de tout polygone {ADMIN_LEVEL} "
            "(généralement une généralisation du trait de côte ou d'un lac). Ils sont conservés "
            "dans les totaux nationaux mais exclus des tableaux infranationaux.", "warn")

# 10 · Construire les indicateurs

Quatre définitions, à recopier telles quelles dans vos métadonnées.

**1 · Couverture de la mesure en population**
$$\text{cov}_{pop} = \frac{\sum_{c \in \mathcal{M}} P_c}{\sum_{c} P_c}$$
où $\mathcal{M}$ est l'ensemble des cellules de population contenant au moins un carreau mesuré.
*Cet indicateur mesure où se trouvent les utilisateurs de Speedtest, pas où se trouve le réseau.*

**2 · Débit descendant moyen pondéré par la population**
$$\bar{S}_{pop} = \frac{\sum_t w_t S_t}{\sum_t w_t}, \qquad w_t = \frac{P_{c(t)}}{n_{c(t)}}$$
La moyenne pondérée par le nombre de carreaux — la moyenne naïve — donne la même voix à chaque
carré de 611 m, qu'il abrite 4 ou 4 000 habitants. Comparez les deux : l'écart est votre message
principal.

**3 · Médiane pondérée par la population** — mêmes poids, mais le 50ᵉ centile de la distribution
pondérée. **Publiez la médiane, pas la moyenne** : les distributions de débits sont fortement
asymétriques à droite et une poignée de carreaux fibrés tire n'importe quelle moyenne vers le haut.

**4 · Part de la population mesurée au-dessus d'un seuil de service**
$$\pi_{\ge x} = \frac{\sum_{t : S_t \ge x} w_t}{\sum_t w_t}$$
calculée à 10 Mbit/s (seuil de référence de l'UIT pour un haut débit utilisable) et à 25 Mbit/s.

In [ ]:
IND_COLS = ["pop_tile", "d_mbps", "u_mbps", "latency_ms", "tests", "devices", "tile_km2"]


def wq(values, weights, q):
    """Quantile pondéré (interpolation linéaire sur le poids cumulé)."""
    v = np.asarray(values, float); w = np.asarray(weights, float)
    m = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if m.sum() == 0:
        return np.nan
    v, w = v[m], w[m]
    o = np.argsort(v); v, w = v[o], w[o]
    cw = np.cumsum(w) - 0.5 * w
    return float(np.interp(q * w.sum(), cw, v))


def indicators(g):
    """Jeu complet d'indicateurs pour un groupe de carreaux (un service, une zone, un trimestre)."""
    w, s = g["pop_tile"].values, g["d_mbps"].values
    W = np.nansum(w)
    out = {
        "tiles":        len(g),
        "tests":        float(g["tests"].sum()),
        "devices":      float(g["devices"].sum()),
        "pop_weight":   float(W),
        "area_km2":     float(g["tile_km2"].sum()),
        "d_mean_tile":  float(np.nanmean(s)),
        "d_median_tile": float(np.nanmedian(s)),
        "d_mean_pop":   float(np.nansum(w * s) / W) if W > 0 else np.nan,
        "d_median_pop": wq(s, w, 0.50),
        "d_p10_pop":    wq(s, w, 0.10),
        "d_p90_pop":    wq(s, w, 0.90),
        "u_median_pop": wq(g["u_mbps"].values, w, 0.50),
        "lat_median_pop": wq(g["latency_ms"].values, w, 0.50),
        "pct_above_bb": float(np.nansum(w[s >= BROADBAND_MBPS]) / W * 100) if W > 0 else np.nan,
        "pct_above_hi": float(np.nansum(w[s >= GOOD_SPEED_MBPS]) / W * 100) if W > 0 else np.nan,
    }
    out["divide_ratio"] = (out["d_p90_pop"] / out["d_p10_pop"]
                           if out["d_p10_pop"] and out["d_p10_pop"] > 0 else np.nan)
    return pd.Series(out)


# ── national, dernier trimestre ────────────────────────────────────────────────
latest = df[(df.year == LATEST_Y) & (df.quarter == LATEST_Q)]
national = latest.groupby("service")[IND_COLS].apply(indicators)
for s in SERVICES:
    if s in national.index:
        cov = cells.loc[cells[f"measured_{s}"], "pop"].sum()
        national.loc[s, "pop_covered"]     = cov
        national.loc[s, "pop_coverage_pct"] = 100 * cov / POP_TOTAL
        national.loc[s, "area_coverage_pct"] = 100 * national.loc[s, "area_km2"] / COUNTRY_AREA_KM2

display(style_table(
    national[["tiles", "tests", "pop_coverage_pct", "area_coverage_pct",
              "d_median_tile", "d_median_pop", "d_mean_pop",
              "pct_above_bb", "pct_above_hi", "divide_ratio", "lat_median_pop"]]
    .rename(columns={
        "tiles": "Carreaux", "tests": "Tests", "pop_coverage_pct": "Pop. mesurée %",
        "area_coverage_pct": "Territoire mesuré %", "d_median_tile": "Médiane (par carreau)",
        "d_median_pop": "Médiane (par habitant)", "d_mean_pop": "Moyenne (par habitant)",
        "pct_above_bb": f"% pop ≥ {BROADBAND_MBPS} Mbit/s",
        "pct_above_hi": f"% pop ≥ {GOOD_SPEED_MBPS} Mbit/s",
        "divide_ratio": "P90/P10", "lat_median_pop": "Latence (ms)"})
    .style.format({"Carreaux": "{:,.0f}", "Tests": "{:,.0f}", "Pop. mesurée %": "{:.1f}",
                   "Territoire mesuré %": "{:.2f}", "Médiane (par carreau)": "{:.1f}",
                   "Médiane (par habitant)": "{:.1f}", "Moyenne (par habitant)": "{:.1f}",
                   f"% pop ≥ {BROADBAND_MBPS} Mbit/s": "{:.1f}",
                   f"% pop ≥ {GOOD_SPEED_MBPS} Mbit/s": "{:.1f}",
                   "P90/P10": "{:.1f}", "Latence (ms)": "{:.0f}"}),
    f"Indicateurs nationaux de connectivité — {COUNTRY_NAME}, {LATEST_Y} T{LATEST_Q} "
    f"(débit descendant, Mbit/s)"))

In [ ]:
# ── tableau infranational ───────────────────────────────────────────────────────
sub = (latest.dropna(subset=["admin_name"])
             .groupby(["service", "admin_name"])[IND_COLS]
             .apply(indicators).reset_index())

pop_adm = cells.groupby("admin_name")["pop"].sum().rename("pop_total").reset_index()

cov_rows = []
for s in SERVICES:
    c = (cells[cells[f"measured_{s}"]].groupby("admin_name")["pop"].sum()
         .rename("pop_covered").reset_index())
    c["service"] = s
    cov_rows.append(c)
sub = sub.merge(pd.concat(cov_rows, ignore_index=True),
                on=["service", "admin_name"], how="left")
sub["pop_covered"] = sub["pop_covered"].fillna(0.0)
sub = sub.merge(pop_adm, on="admin_name", how="left")
sub = sub.merge(admin[["admin_name", "area_km2"]].rename(columns={"area_km2": "adm_km2"}),
                on="admin_name", how="left")
sub["pop_coverage_pct"]  = 100 * sub["pop_covered"] / sub["pop_total"]
sub["area_coverage_pct"] = 100 * sub["area_km2"] / sub["adm_km2"]
sub["density"]           = sub["pop_total"] / sub["adm_km2"]

MAIN_SERVICE = "mobile" if "mobile" in SERVICES else SERVICES[0]
main = sub[sub.service == MAIN_SERVICE].sort_values("d_median_pop", ascending=False)

show = main[["admin_name", "pop_total", "pop_coverage_pct", "tests",
             "d_median_pop", "pct_above_bb", "divide_ratio", "lat_median_pop"]].copy()
show.columns = [ADMIN_LEVEL, "Population", "Pop. mesurée %", "Tests",
                "Mbit/s médians (par habitant)", f"% pop ≥ {BROADBAND_MBPS} Mbit/s",
                "P90/P10", "Latence ms"]
display(style_table(show.head(20).style.format({
    "Population": "{:,.0f}", "Pop. mesurée %": "{:.1f}", "Tests": "{:,.0f}",
    "Mbit/s médians (par habitant)": "{:.1f}", f"% pop ≥ {BROADBAND_MBPS} Mbit/s": "{:.1f}",
    "P90/P10": "{:.1f}", "Latence ms": "{:.0f}"}),
    f"Connectivité {MAIN_SERVICE} par {ADMIN_LEVEL} — "
    f"{LATEST_Y} T{LATEST_Q} (20 premières selon la médiane pondérée)"))

In [ ]:
# ── type d'habitat (urbain / périurbain / rural) ────────────────────────────
settle = (latest.groupby(["service", "settlement"])[IND_COLS]
                .apply(indicators).reset_index())
pop_settle = cells.groupby("settlement")["pop"].sum().rename("pop_total").reset_index()
settle = settle.merge(pop_settle, on="settlement", how="left")
for s in SERVICES:
    for st in settle["settlement"].unique():
        m = cells[(cells.settlement == st) & (cells[f"measured_{s}"])]["pop"].sum()
        settle.loc[(settle.service == s) & (settle.settlement == st), "pop_covered"] = m
settle["pop_coverage_pct"] = 100 * settle["pop_covered"] / settle["pop_total"]

order = ["Urban", "Peri-urban", "Rural"]
settle["settlement"] = pd.Categorical(settle["settlement"], order, ordered=True)
settle = settle.sort_values(["service", "settlement"])

display(style_table(
    settle[settle.service == MAIN_SERVICE][
        ["settlement", "pop_total", "pop_coverage_pct", "tiles",
         "d_median_pop", "pct_above_bb"]]
    .rename(columns={"settlement": "Type d'habitat", "pop_total": "Population",
                     "pop_coverage_pct": "Pop. mesurée %", "tiles": "Carreaux mesurés",
                     "d_median_pop": "Mbit/s médians (par habitant)",
                     "pct_above_bb": f"% pop ≥ {BROADBAND_MBPS} Mbit/s"})
    .style.format({"Population": "{:,.0f}", "Pop. mesurée %": "{:.1f}",
                   "Carreaux mesurés": "{:,.0f}", "Mbit/s médians (par habitant)": "{:.1f}",
                   f"% pop ≥ {BROADBAND_MBPS} Mbit/s": "{:.1f}"}),
    f"La fracture urbain–rural — {MAIN_SERVICE}, {LATEST_Y} T{LATEST_Q} "
    f"(urbain ≥ {URBAN_DENS_MIN}/km², rural < {PERIURBAN_DENS_MIN}/km²)"))

# 11 · Sortie bilingue

Ce notebook est rédigé en français, mais le **tableau de bord publié est bilingue** : chaque
libellé, titre, enseignement, légende, infobulle et note méthodologique existe en français et en
anglais, et le lecteur bascule d'une langue à l'autre d'un clic. Le choix est mémorisé dans le
navigateur, et la page s'ouvre dans la langue du lecteur lorsqu'elle peut la détecter.

Tout ce qui est traduisible est rassemblé dans l'unique dictionnaire ci-dessous. Ajouter une
troisième langue consiste à ajouter une entrée à `LANGS` et un bloc à `TXT` — rien d'autre ne change
dans le notebook.

> **Pourquoi cela compte ici.** Un indicateur de connectivité qui n'existe qu'en anglais est un
> indicateur que la moitié des bureaux statistiques du continent ne peut pas faire circuler en
> interne. Produire les deux versions à partir du même calcul garantit en outre qu'elles ne
> pourront jamais diverger : il y a un seul chiffre, restitué deux fois.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Catalogue bilingue — chaque fragment de texte qui parvient au lecteur
# ══════════════════════════════════════════════════════════════════════════════
LANGS = ["fr", "en"]          # ajoutez une langue ici + un bloc dans TXT ci-dessous
NB_LANG = "fr"                # langue utilisée pour les aperçus à l'intérieur du notebook

TXT = {
"en": {
  # ---- chrome -------------------------------------------------------------
  "lang_name": "English", "other_lang_btn": "Français",
  "kicker": "AfDB · AU STATAFRIC · STG17 &nbsp;·&nbsp; {period}",
  "title": "{country} — Connectivity and Population",
  "subtitle": ("Ookla® Speedtest Open Data joined to WorldPop gridded population — "
               "a population-weighted connectivity indicator for every {level} unit"),
  "meta_desc": ("Population-weighted connectivity indicators for {country} from Ookla Speedtest "
                "Open Data and WorldPop, {period}."),
  "nav_kpi": "Key figures", "nav_ins": "Insights", "nav_maps": "Maps",
  "nav_charts": "Charts", "nav_table": "Table", "nav_method": "Method &amp; limitations",
  # ---- section 1 ----------------------------------------------------------
  "s1_kick": "01 · Key figures", "s1_title": "{country} at a glance",
  "s1_lede": ("{tests} {service} tests recorded in {tiles} measured squares, weighted by the "
              "population of {pop} inhabitants."),
  "s1_src": ("Reference period {period}. Figures “per person” are weighted by WorldPop {wpyear} "
             "gridded population; figures “per tile” give every measured square equal weight."),
  "kpi_pop": "Population", "kpi_med_person": "Median download<br>per person",
  "kpi_med_tile": "Median download<br>per tile", "kpi_above": "Population ≥ {bb} Mbps",
  "kpi_pop_meas": "Population measured", "kpi_area_meas": "Land area measured",
  "kpi_gini": "Connectivity Gini", "kpi_lat": "Median latency",
  "u_mbps": "Mbps", "u_ms": "ms", "u_pct": "%", "u_pct_meas": "% of measured", "u_wp": "WorldPop {y}",
  # ---- section 2 ----------------------------------------------------------
  "s2_kick": "02 · What the data says", "s2_title": "Automated findings",
  "s2_lede": "Generated directly from the computation — no figure in this section was typed by hand.",
  # ---- section 3 ----------------------------------------------------------
  "s3_kick": "03 · Geography", "s3_title": "Where connectivity is, and where measurement is not",
  "s3_lede": ("The first map answers “how fast”, the second “where exactly was it measured”, and "
              "the third — the one that usually changes the conversation — “who is missing from "
              "the sample”."),
  "map1_cap": "Map 1 — Population-weighted median download speed by {level}.",
  "map2_cap": ("Map 2 — The measurement grid itself. Use the layer control, top right, "
               "for test density."),
  "map3_cap": ("Map 3 — Populated cells with no measurement in the reference quarter. "
               "A sampling map, not a network coverage map."),
  # ---- section 4 ----------------------------------------------------------
  "s4_kick": "04 · Analysis", "s4_title": "Six views of the same question",
  "s4_lede": "Hover any chart for the underlying values.",
  # ---- section 5 ----------------------------------------------------------
  "s5_kick": "05 · Reference", "s5_title": "Indicators by {level}",
  "s5_lede": ("Also available as CSV and GeoJSON in this repository. “Per person” is the figure to "
              "quote; “per tile” is shown so that the difference stays visible."),
  "th_unit": "{level}", "th_pop": "Population", "th_meas": "Pop. measured", "th_tests": "Tests",
  "th_med_person": "Median Mbps<br>per person", "th_med_tile": "Median Mbps<br>per tile",
  "th_above": "% pop ≥ {bb} Mbps", "th_ratio": "P90/P10", "th_lat": "Latency ms",
  # ---- section 6 ----------------------------------------------------------
  "s6_kick": "06 · Documentation", "s6_title": "Method, sources and limitations",
  "h_sources": "Sources", "h_method": "Method",
  "h_limits": "Limitations — read before quoting any figure", "h_repro": "Reproducibility",
  "src_ookla": ("<b>Ookla® Speedtest Open Data</b> — performance tiles, Web-Mercator zoom 16 "
                "(≈611 m at the equator), {period}. Licence <b>CC BY-NC-SA 4.0</b>."),
  "src_wp": ("<b>WorldPop</b> — global gridded population {wpyear}, {wpres}, UN-adjusted. "
             "Licence CC BY 4.0."),
  "src_gb": ("<b>geoBoundaries</b> (gbOpen) — administrative boundaries {level}. "
             "Licence CC BY 4.0."),
  "meth_1": ("Tiles extracted from the global quarterly parquet files by quadkey range predicate, "
             "then clipped to the national polygon on the tile centroid."),
  "meth_2": ("Each tile is located in the WorldPop grid by its centroid; the population of a cell "
             "is shared equally among the tiles it contains, giving each tile a population weight."),
  "meth_3": ("Headline speeds are <b>population-weighted medians</b>: the median of the "
             "distribution in which each tile carries the weight of the people it represents."),
  "meth_4": ("Settlement classes are a density proxy: urban ≥ {urb} people/km², "
             "peri-urban ≥ {peri}, rural below."),
  "meth_5": ("Coverage of measurement = share of the national population living in a 1 km cell "
             "containing at least one measured tile."),
  "repro_1": ("Produced by the notebook "
              "<code>02-Lab-Ookla-Speedtest-Open-Data-and-WorldPop.ipynb</code>, "
              "generated on {date} for <b>{country} ({iso3})</b>."),
  "repro_2": ("Re-run it with a different <code>COUNTRY_ISO3</code> to rebuild the whole product "
              "for another country."),
  "footer": ("<b>{country} — Connectivity and population</b><br>"
             "African Development Bank · AU STATAFRIC — STG17.<br>"
             "Contains information from <b>Ookla® Speedtest Open Data</b>, used under "
             "<a href='https://creativecommons.org/licenses/by-nc-sa/4.0/'>CC BY-NC-SA 4.0</a>. "
             "Ookla trademarks are the property of Ookla, LLC; this product is not endorsed by or "
             "affiliated with Ookla. Derivative works must carry the same licence and must not be "
             "used commercially.<br>"
             "Population data © WorldPop (CC BY 4.0) · Boundaries © geoBoundaries (CC BY 4.0)."),
  # ---- charts -------------------------------------------------------------
  "src_note": ("Source: Ookla® Speedtest Open Data ({period}, CC BY-NC-SA 4.0) · "
               "WorldPop {wpyear} ({wpres}, CC BY 4.0) · geoBoundaries gbOpen"),
  "c1_title": "How fast is a measured square?",
  "c1_sub": ("Distribution of tile-level download speeds, 99th percentile clipped. "
             "The long right tail is why we publish medians."),
  "c1_x": "Download speed (Mbps)", "c1_y": "Number of tiles", "c1_tiles": "tiles",
  "c2_title": "Where is {service} connectivity strongest?",
  "c2_sub": "Population-weighted median download speed by {level}, {period}",
  "c2_x": "Population-weighted median download speed (Mbps)", "c2_nat": "national",
  "c2_h": ("Median (per person)", "Population", "Population measured", "Tests"),
  "c3_title": "Does density buy speed?",
  "c3_sub": "Each bubble is one {level} unit; size = population, colour = share of population measured",
  "c3_x": "Population density (people / km², log scale)", "c3_y": "Median download speed (Mbps)",
  "c3_cbar": "Pop.<br>measured %", "c3_h": ("Density", "Median", "Population"),
  "c3_r": "(log density)",
  "c4_title": "Is connectivity improving?",
  "c4_sub": "Population-weighted median speed (lines) and measurement volume (bars)",
  "c4_y": "Population-weighted median (Mbps)", "c4_y2": "Tiles measured",
  "c4_line": "{service} — median Mbps", "c4_bar": "{service} — tiles measured",
  "c5_title": "How unequally is bandwidth distributed?",
  "c5_sub": ("Connectivity Lorenz curve — Gini coefficient <b>{gini}</b> "
             "(0 = every person experiences the same speed, 1 = total concentration)"),
  "c5_x": "Cumulative share of the measured population (%)",
  "c5_y": "Cumulative share of measured bandwidth (%)",
  "c5_eq": "perfect equality", "c5_obs": "observed",
  "c5_h": "Poorest-served %{x:.0f}% of people<br>hold %{y:.0f}% of measured bandwidth",
  "c5_a": "<b>{v}%</b> of bandwidth<br>for {q}% of people",
  "c6_title": "Two divides, not one",
  "c6_sub": ("Speed gap (bars) and measurement gap (dotted line) between urban, peri-urban "
             "and rural areas"),
  "c6_y": "Median download speed (Mbps)", "c6_y2": "% of population measured",
  "c6_cov": "% of population measured",
  # ---- maps ---------------------------------------------------------------
  "mp_layer_speed": "{service} median speed", "mp_layer_grid": "{service} speed grid",
  "mp_layer_heat": "Test density (heatmap)", "mp_boundary": "National boundary",
  "mp_cbar1": "Population-weighted median {service} download speed (Mbps)",
  "mp_cbar2": "Download speed (Mbps) — {cell}",
  "mp_native": "native z16 tiles (≈611 m)", "mp_agg": "aggregated to z{z} (≈{km} km)",
  "mp_t_unit": "{level}:", "mp_t_pop": "Population:", "mp_t_meas": "Population measured (%):",
  "mp_t_person": "Median Mbps (per person):", "mp_t_tile": "Median Mbps (per tile):",
  "mp_t_above": "% pop ≥ {bb} Mbps:", "mp_t_lat": "Latency (ms):", "mp_t_tests": "Tests:",
  "mp_t_speed": "Download (Mbps):", "mp_t_estpop": "Est. population:", "mp_t_z16": "z16 tiles:",
  "mp_m1_title": "{service} connectivity by {level}",
  "mp_m3_legend": "Measurement gap", "mp_m3_a": "Dense unmeasured population",
  "mp_m3_b": "Unmeasured population", "mp_m3_c": "Measured population (hidden layer)",
  "mp_m3_note": ("{pop} people live in a 1 km cell with no Speedtest measurement in {period}."),
  "mp_layer_gap": "Unmeasured population", "mp_layer_meas": "Measured population",
  # ---- settlement classes -------------------------------------------------
  "Urban": "Urban", "Peri-urban": "Peri-urban", "Rural": "Rural",
},

"fr": {
  # ---- chrome -------------------------------------------------------------
  "lang_name": "Français", "other_lang_btn": "English",
  "kicker": "BAD · UA STATAFRIC · STG17 &nbsp;·&nbsp; {period}",
  "title": "{country} — Connectivité et population",
  "subtitle": ("Données ouvertes Ookla® Speedtest croisées avec la grille de population WorldPop — "
               "un indicateur de connectivité pondéré par la population pour chaque unité {level}"),
  "meta_desc": ("Indicateurs de connectivité pondérés par la population pour {country}, à partir "
                "des données ouvertes Ookla Speedtest et de WorldPop, {period}."),
  "nav_kpi": "Chiffres clés", "nav_ins": "Enseignements", "nav_maps": "Cartes",
  "nav_charts": "Graphiques", "nav_table": "Tableau", "nav_method": "Méthode et limites",
  # ---- section 1 ----------------------------------------------------------
  "s1_kick": "01 · Chiffres clés", "s1_title": "{country} en un coup d’œil",
  "s1_lede": ("{tests} tests {service} enregistrés dans {tiles} carreaux mesurés, pondérés par la "
              "population de {pop} habitants."),
  "s1_src": ("Période de référence {period}. Les valeurs « par habitant » sont pondérées par la "
             "grille de population WorldPop {wpyear} ; les valeurs « par carreau » donnent le même "
             "poids à chaque carreau mesuré."),
  "kpi_pop": "Population", "kpi_med_person": "Débit descendant<br>médian par habitant",
  "kpi_med_tile": "Débit descendant<br>médian par carreau", "kpi_above": "Population ≥ {bb} Mbit/s",
  "kpi_pop_meas": "Population mesurée", "kpi_area_meas": "Territoire mesuré",
  "kpi_gini": "Gini de connectivité", "kpi_lat": "Latence médiane",
  "u_mbps": "Mbit/s", "u_ms": "ms", "u_pct": "%", "u_pct_meas": "% des personnes mesurées",
  "u_wp": "WorldPop {y}",
  # ---- section 2 ----------------------------------------------------------
  "s2_kick": "02 · Ce que disent les données", "s2_title": "Enseignements générés automatiquement",
  "s2_lede": ("Produits directement par le calcul — aucun chiffre de cette section n’a été saisi "
              "à la main."),
  # ---- section 3 ----------------------------------------------------------
  "s3_kick": "03 · Géographie",
  "s3_title": "Où se trouve la connectivité, et où la mesure fait défaut",
  "s3_lede": ("La première carte répond à « à quelle vitesse », la deuxième à « où exactement la "
              "mesure a-t-elle eu lieu », et la troisième — celle qui change généralement la "
              "discussion — à « qui est absent de l’échantillon »."),
  "map1_cap": "Carte 1 — Débit descendant médian pondéré par la population, par unité {level}.",
  "map2_cap": ("Carte 2 — La grille de mesure elle-même. Le sélecteur de couches, en haut à "
               "droite, donne accès à la densité de tests."),
  "map3_cap": ("Carte 3 — Cellules peuplées sans aucune mesure sur le trimestre de référence. "
               "C’est une carte d’échantillonnage, pas une carte de couverture réseau."),
  # ---- section 4 ----------------------------------------------------------
  "s4_kick": "04 · Analyse", "s4_title": "Six regards sur la même question",
  "s4_lede": "Survolez un graphique pour afficher les valeurs sous-jacentes.",
  # ---- section 5 ----------------------------------------------------------
  "s5_kick": "05 · Référence", "s5_title": "Indicateurs par unité {level}",
  "s5_lede": ("Également disponibles en CSV et GeoJSON dans ce dépôt. La valeur « par habitant » "
              "est celle qu’il faut citer ; la valeur « par carreau » est affichée pour que "
              "l’écart reste visible."),
  "th_unit": "{level}", "th_pop": "Population", "th_meas": "Pop. mesurée", "th_tests": "Tests",
  "th_med_person": "Mbit/s médians<br>par habitant", "th_med_tile": "Mbit/s médians<br>par carreau",
  "th_above": "% pop ≥ {bb} Mbit/s", "th_ratio": "P90/P10", "th_lat": "Latence ms",
  # ---- section 6 ----------------------------------------------------------
  "s6_kick": "06 · Documentation", "s6_title": "Méthode, sources et limites",
  "h_sources": "Sources", "h_method": "Méthode",
  "h_limits": "Limites — à lire avant de citer le moindre chiffre",
  "h_repro": "Reproductibilité",
  "src_ookla": ("<b>Ookla® Speedtest Open Data</b> — carreaux de performance, zoom 16 en "
                "projection Web-Mercator (≈611 m à l’équateur), {period}. "
                "Licence <b>CC BY-NC-SA 4.0</b>."),
  "src_wp": ("<b>WorldPop</b> — grille mondiale de population {wpyear}, {wpres}, ajustée aux "
             "estimations des Nations unies. Licence CC BY 4.0."),
  "src_gb": ("<b>geoBoundaries</b> (gbOpen) — limites administratives {level}. "
             "Licence CC BY 4.0."),
  "meth_1": ("Carreaux extraits des fichiers parquet trimestriels mondiaux par prédicat "
             "d’intervalle sur le quadkey, puis découpés sur le polygone national selon le "
             "centroïde du carreau."),
  "meth_2": ("Chaque carreau est localisé dans la grille WorldPop par son centroïde ; la "
             "population d’une cellule est répartie à parts égales entre les carreaux qu’elle "
             "contient, ce qui donne à chaque carreau un poids de population."),
  "meth_3": ("Les débits mis en avant sont des <b>médianes pondérées par la population</b> : la "
             "médiane de la distribution dans laquelle chaque carreau porte le poids des "
             "habitants qu’il représente."),
  "meth_4": ("Les classes d’habitat sont une approximation par la densité : urbain ≥ {urb} "
             "habitants/km², périurbain ≥ {peri}, rural en dessous."),
  "meth_5": ("Couverture de la mesure = part de la population nationale vivant dans une cellule "
             "de 1 km contenant au moins un carreau mesuré."),
  "repro_1": ("Produit par le notebook "
              "<code>02-Lab-Ookla-Speedtest-Open-Data-and-WorldPop.ipynb</code>, "
              "généré le {date} pour <b>{country} ({iso3})</b>."),
  "repro_2": ("Relancez-le avec un autre <code>COUNTRY_ISO3</code> pour reconstruire l’ensemble du "
              "produit pour un autre pays."),
  "footer": ("<b>{country} — Connectivité et population</b><br>"
             "Banque africaine de développement · UA STATAFRIC — STG17.<br>"
             "Contient des informations issues des <b>données ouvertes Ookla® Speedtest</b>, "
             "utilisées sous licence "
             "<a href='https://creativecommons.org/licenses/by-nc-sa/4.0/deed.fr'>CC BY-NC-SA 4.0</a>. "
             "Les marques Ookla sont la propriété d’Ookla, LLC ; ce produit n’est ni approuvé par "
             "Ookla ni affilié à Ookla. Toute œuvre dérivée doit porter la même licence et ne peut "
             "faire l’objet d’un usage commercial.<br>"
             "Données de population © WorldPop (CC BY 4.0) · Limites © geoBoundaries (CC BY 4.0)."),
  # ---- charts -------------------------------------------------------------
  "src_note": ("Sources : Ookla® Speedtest Open Data ({period}, CC BY-NC-SA 4.0) · "
               "WorldPop {wpyear} ({wpres}, CC BY 4.0) · geoBoundaries gbOpen"),
  "c1_title": "À quelle vitesse va un carreau mesuré ?",
  "c1_sub": ("Distribution des débits descendants par carreau, écrêtée au 99ᵉ centile. "
             "C’est cette longue queue à droite qui impose de publier des médianes."),
  "c1_x": "Débit descendant (Mbit/s)", "c1_y": "Nombre de carreaux", "c1_tiles": "carreaux",
  "c2_title": "Où la connectivité {service} est-elle la meilleure ?",
  "c2_sub": ("Débit descendant médian pondéré par la population, par unité {level}, {period}"),
  "c2_x": "Débit descendant médian pondéré par la population (Mbit/s)", "c2_nat": "national",
  "c2_h": ("Médiane (par habitant)", "Population", "Population mesurée", "Tests"),
  "c3_title": "La densité achète-t-elle du débit ?",
  "c3_sub": ("Chaque bulle est une unité {level} ; taille = population, "
             "couleur = part de la population mesurée"),
  "c3_x": "Densité de population (habitants / km², échelle logarithmique)",
  "c3_y": "Débit descendant médian (Mbit/s)",
  "c3_cbar": "Pop.<br>mesurée %", "c3_h": ("Densité", "Médiane", "Population"),
  "c3_r": "(densité en log)",
  "c4_title": "La connectivité progresse-t-elle ?",
  "c4_sub": ("Débit médian pondéré par la population (courbes) et volume de mesure (barres)"),
  "c4_y": "Médiane pondérée par la population (Mbit/s)", "c4_y2": "Carreaux mesurés",
  "c4_line": "{service} — Mbit/s médians", "c4_bar": "{service} — carreaux mesurés",
  "c5_title": "La bande passante est-elle répartie inégalement ?",
  "c5_sub": ("Courbe de Lorenz de la connectivité — coefficient de Gini <b>{gini}</b> "
             "(0 = tout le monde connaît le même débit, 1 = concentration totale)"),
  "c5_x": "Part cumulée de la population mesurée (%)",
  "c5_y": "Part cumulée de la bande passante mesurée (%)",
  "c5_eq": "égalité parfaite", "c5_obs": "observé",
  "c5_h": ("Les %{x:.0f}% les moins bien servis<br>détiennent %{y:.0f}% de la bande passante mesurée"),
  "c5_a": "<b>{v}%</b> de la bande passante<br>pour {q}% des habitants",
  "c6_title": "Deux fractures, et non une seule",
  "c6_sub": ("Écart de débit (barres) et écart de mesure (pointillés) entre zones urbaines, "
             "périurbaines et rurales"),
  "c6_y": "Débit descendant médian (Mbit/s)", "c6_y2": "% de la population mesurée",
  "c6_cov": "% de la population mesurée",
  # ---- maps ---------------------------------------------------------------
  "mp_layer_speed": "Débit médian {service}", "mp_layer_grid": "Grille des débits {service}",
  "mp_layer_heat": "Densité de tests (carte de chaleur)", "mp_boundary": "Frontière nationale",
  "mp_cbar1": "Débit descendant {service} médian pondéré par la population (Mbit/s)",
  "mp_cbar2": "Débit descendant (Mbit/s) — {cell}",
  "mp_native": "carreaux z16 natifs (≈611 m)", "mp_agg": "agrégés au z{z} (≈{km} km)",
  "mp_t_unit": "{level} :", "mp_t_pop": "Population :", "mp_t_meas": "Population mesurée (%) :",
  "mp_t_person": "Mbit/s médians (par habitant) :", "mp_t_tile": "Mbit/s médians (par carreau) :",
  "mp_t_above": "% pop ≥ {bb} Mbit/s :", "mp_t_lat": "Latence (ms) :", "mp_t_tests": "Tests :",
  "mp_t_speed": "Débit descendant (Mbit/s) :", "mp_t_estpop": "Population estimée :",
  "mp_t_z16": "Carreaux z16 :",
  "mp_m1_title": "Connectivité {service} par unité {level}",
  "mp_m3_legend": "Déficit de mesure", "mp_m3_a": "Population non mesurée, dense",
  "mp_m3_b": "Population non mesurée", "mp_m3_c": "Population mesurée (couche masquée)",
  "mp_m3_note": ("{pop} habitants vivent dans une cellule de 1 km sans aucune mesure Speedtest "
                 "sur {period}."),
  "mp_layer_gap": "Population non mesurée", "mp_layer_meas": "Population mesurée",
  # ---- settlement classes -------------------------------------------------
  "Urban": "Urbain", "Peri-urban": "Périurbain", "Rural": "Rural",
},
}


def T(key, lang, **kw):
    """Traduit `key` dans `lang` et remplit ses champs."""
    s = TXT[lang][key]
    return s.format(**kw) if kw else s


# valeurs injectées dans presque toutes les chaînes
def ctx(lang, **extra):
    d = dict(country=COUNTRY_NAME, iso3=COUNTRY_ISO3, level=ADMIN_LEVEL,
             period=f"{LATEST_Y} Q{LATEST_Q}", service=MAIN_SERVICE.capitalize(),
             bb=BROADBAND_MBPS, wpyear=WORLDPOP_YEAR, wpres=WORLDPOP_RES,
             urb=URBAN_DENS_MIN, peri=PERIURBAN_DENS_MIN)
    d.update(extra)
    return d


missing = {l: [k for k in TXT["en"] if k not in TXT[l]] for l in LANGS}
for l, ks in missing.items():
    if ks:
        print(f"ATTENTION — il manque {len(ks)} clés à {l} : {ks[:6]}")
print(f"Langues : {', '.join(TXT[l]['lang_name'] for l in LANGS)}  ·  "
      f"{len(TXT['en'])} chaînes chacune  ·  aperçus du notebook en « {NB_LANG} »")

# 12 · Enseignements automatiques, dans les deux langues

Tout ce qui suit est **calculé, et non rédigé**. Relancez le notebook pour un autre pays et les
phrases changent en conséquence. C'est délibéré : un enseignement que l'on peut générer est un
enseignement que l'on peut auditer, et c'est la partie que les lecteurs de votre tableau de bord
liront réellement.

Chaque constat est produit en français et en anglais **à partir des mêmes chiffres**, si bien que
les deux versions ne peuvent pas se contredire. Les nombres sont formatés selon les conventions de
chaque langue — le français utilise la virgule décimale, une espace fine insécable pour les
milliers, et une espace avant le signe pourcent. Se tromper sur ce détail est précisément ce qui
signale à un lecteur que sa version a été traduite après coup.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Formatage des nombres selon la langue
# ══════════════════════════════════════════════════════════════════════════════
NBSP = "\u202f"          # espace fine insécable, séparateur des milliers en français

def num(x, d=1, lang="en"):
    """Formate un nombre selon les conventions de la langue cible."""
    if x is None or not np.isfinite(x):
        return "–"
    s = f"{x:,.{d}f}"
    if lang == "fr":
        s = s.replace(",", "\x00").replace(".", ",").replace("\x00", NBSP)
    return s

def pct(x, d=1, lang="en"):
    return num(x, d, lang) + ("{}%".format(NBSP) if lang == "fr" else "%")

def big(x, lang="en", d=1):
    """Forme compacte : 1.9 M / 1,9 M, 12.4 k / 12,4 k."""
    if x is None or not np.isfinite(x):
        return "–"
    a = abs(x)
    if a >= 1e9:
        return num(x / 1e9, d, lang) + (f"{NBSP}Md" if lang == "fr" else " bn")
    if a >= 1e6:
        return num(x / 1e6, d, lang) + (f"{NBSP}M" if lang == "fr" else " M")
    if a >= 1e3:
        return num(x / 1e3, d, lang) + (f"{NBSP}k" if lang == "fr" else " k")
    return num(x, d, lang)

def mbps(x, lang="en", d=1):
    return num(x, d, lang) + (f"{NBSP}Mbit/s" if lang == "fr" else " Mbps")

print("en:", big(1938472, "en"), "·", pct(13.54, 1, "en"), "·", mbps(24.7, "en"))
print("fr:", big(1938472, "fr"), "·", pct(13.54, 1, "fr"), "·", mbps(24.7, "fr"))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Les constats — un seul calcul, deux langues
# ══════════════════════════════════════════════════════════════════════════════
INSIGHTS = []

def add(kind, en_title, en_text, fr_title, fr_text):
    INSIGHTS.append({"kind": kind,
                     "en": {"title": en_title, "text": en_text},
                     "fr": {"title": fr_title, "text": fr_text}})

nat = national.loc[MAIN_SERVICE]
SVC_FR = {"mobile": "mobile", "fixed": "fixe"}[MAIN_SERVICE]

# ── 1 · l'écart de pondération ──────────────────────────────────────────────────
gap = nat["d_median_pop"] - nat["d_median_tile"]
rel = abs(gap) / max(nat["d_median_tile"], 1e-9) * 100
add("info",
    "Weighting changes the headline",
    f"The median {MAIN_SERVICE} download speed is <b>{mbps(nat['d_median_tile'])} per measured "
    f"tile</b> but <b>{mbps(nat['d_median_pop'])} per person</b> — a gap of "
    f"{'+' if gap > 0 else '−'}{mbps(abs(gap))} ({num(rel, 0)}%). "
    + ("Population weighting raises the figure: the better-served squares are also the more "
       "densely populated ones." if gap > 0 else
       "Population weighting lowers the figure: the fastest squares are sparsely populated, so a "
       "tile average flatters the national picture."),
    "La pondération change le chiffre à retenir",
    f"Le débit descendant médian {SVC_FR} est de <b>{mbps(nat['d_median_tile'], 'fr')} par carreau "
    f"mesuré</b> mais de <b>{mbps(nat['d_median_pop'], 'fr')} par habitant</b> — un écart de "
    f"{'+' if gap > 0 else '−'}{mbps(abs(gap), 'fr')} ({pct(rel, 0, 'fr')}). "
    + ("La pondération par la population relève le chiffre : les carreaux les mieux servis sont "
       "aussi les plus densément peuplés." if gap > 0 else
       "La pondération par la population abaisse le chiffre : les carreaux les plus rapides sont "
       "peu peuplés, si bien qu’une moyenne par carreau flatte la situation nationale."))

# ── 2 · ce que la source peut et ne peut pas dire ─────────────────────────────────
unmeasured = POP_TOTAL * (1 - nat["pop_coverage_pct"] / 100)
add("warn",
    "What the source can and cannot tell you",
    f"Measurements exist for <b>{pct(nat['pop_coverage_pct'])} of the population</b> but only "
    f"<b>{pct(nat['area_coverage_pct'], 2)} of the land area</b>. Every figure in this dashboard "
    f"describes the measured population; the remaining {big(unmeasured)} inhabitants are outside "
    "the sample — they are not necessarily unconnected.",
    "Ce que la source peut dire, et ce qu’elle ne peut pas dire",
    f"Des mesures existent pour <b>{pct(nat['pop_coverage_pct'], 1, 'fr')} de la population</b> "
    f"mais seulement <b>{pct(nat['area_coverage_pct'], 2, 'fr')} du territoire</b>. Tous les "
    "chiffres de ce tableau de bord décrivent la population mesurée ; les "
    f"{big(unmeasured, 'fr')} habitants restants sont hors échantillon — ils ne sont pas pour "
    "autant dépourvus de connexion.")

# ── 3 · qualité de service ────────────────────────────────────────────────────
add("info",
    f"Service quality against the {BROADBAND_MBPS} Mbps reference",
    f"<b>{pct(nat['pct_above_bb'])}</b> of the measured population lives where {MAIN_SERVICE} "
    f"download speeds reach at least {BROADBAND_MBPS} Mbps, and <b>{pct(nat['pct_above_hi'])}</b> "
    f"reach {GOOD_SPEED_MBPS} Mbps. Median latency is {num(nat['lat_median_pop'], 0)} ms.",
    f"Qualité de service au regard du seuil de {BROADBAND_MBPS} Mbit/s",
    f"<b>{pct(nat['pct_above_bb'], 1, 'fr')}</b> de la population mesurée vit là où le débit "
    f"descendant {SVC_FR} atteint au moins {BROADBAND_MBPS} Mbit/s, et "
    f"<b>{pct(nat['pct_above_hi'], 1, 'fr')}</b> atteint {GOOD_SPEED_MBPS} Mbit/s. "
    f"La latence médiane est de {num(nat['lat_median_pop'], 0, 'fr')} ms.")

# ── 4 · dispersion infranationale ─────────────────────────────────────────────────
if len(main) >= 2:
    top, bot = main.iloc[0], main.iloc[-1]
    ratio = top["d_median_pop"] / max(bot["d_median_pop"], 1e-9)
    add("warn" if ratio > 3 else "info",
        "Subnational spread",
        f"<b>{top['admin_name']}</b> records {mbps(top['d_median_pop'])} against "
        f"<b>{bot['admin_name']}</b> at {mbps(bot['d_median_pop'])} — a ratio of "
        f"<b>{num(ratio)}×</b> across {len(main)} {ADMIN_LEVEL} units. The within-country "
        f"P90/P10 ratio is {num(nat['divide_ratio'])}×.",
        "Écarts infranationaux",
        f"<b>{top['admin_name']}</b> enregistre {mbps(top['d_median_pop'], 'fr')} contre "
        f"<b>{bot['admin_name']}</b> à {mbps(bot['d_median_pop'], 'fr')} — un rapport de "
        f"<b>{num(ratio, 1, 'fr')}×</b> sur {len(main)} unités {ADMIN_LEVEL}. Le rapport "
        f"P90/P10 à l’intérieur du pays est de {num(nat['divide_ratio'], 1, 'fr')}×.")

# ── 5 · la fracture urbain-rural ─────────────────────────────────────────────
try:
    sm = settle[settle.service == MAIN_SERVICE].set_index("settlement")
    if "Urban" in sm.index and "Rural" in sm.index:
        u, r = sm.loc["Urban"], sm.loc["Rural"]
        sgap = u["d_median_pop"] / max(r["d_median_pop"], 1e-9)
        mgap = u["pop_coverage_pct"] / max(r["pop_coverage_pct"], 1e-9)
        add("risk" if sgap > 2 else "warn",
            "The urban–rural divide",
            f"Urban areas: <b>{mbps(u['d_median_pop'])}</b> with {pct(u['pop_coverage_pct'], 0)} "
            f"of their population measured. Rural areas: <b>{mbps(r['d_median_pop'])}</b> with "
            f"only {pct(r['pop_coverage_pct'], 0)} measured. The speed gap is "
            f"<b>{num(sgap)}×</b>; the <i>measurement</i> gap is {num(mgap)}× and is itself a "
            "finding.",
            "La fracture urbain–rural",
            f"Zones urbaines : <b>{mbps(u['d_median_pop'], 'fr')}</b>, avec "
            f"{pct(u['pop_coverage_pct'], 0, 'fr')} de leur population mesurée. Zones rurales : "
            f"<b>{mbps(r['d_median_pop'], 'fr')}</b>, avec seulement "
            f"{pct(r['pop_coverage_pct'], 0, 'fr')} de mesure. L’écart de débit est de "
            f"<b>{num(sgap, 1, 'fr')}×</b> ; l’écart de <i>mesure</i> est de "
            f"{num(mgap, 1, 'fr')}× et constitue en soi un résultat.")
except Exception:
    pass

# ── 6 · fixe et mobile ────────────────────────────────────────────────
if {"fixed", "mobile"} <= set(national.index):
    f_, m_ = national.loc["fixed"], national.loc["mobile"]
    mobile_first = m_["pop_coverage_pct"] > 2 * f_["pop_coverage_pct"]
    add("info",
        "Fixed versus mobile",
        f"Fixed broadband reaches <b>{mbps(f_['d_median_pop'])}</b> per person against "
        f"<b>{mbps(m_['d_median_pop'])}</b> for mobile, but is measured for only "
        f"{pct(f_['pop_coverage_pct'])} of the population against {pct(m_['pop_coverage_pct'])} "
        "for mobile. In this country, mobile is "
        + ("the de facto access technology." if mobile_first
           else "widely complemented by fixed access."),
        "Fixe et mobile",
        f"Le haut débit fixe atteint <b>{mbps(f_['d_median_pop'], 'fr')}</b> par habitant contre "
        f"<b>{mbps(m_['d_median_pop'], 'fr')}</b> pour le mobile, mais n’est mesuré que pour "
        f"{pct(f_['pop_coverage_pct'], 1, 'fr')} de la population contre "
        f"{pct(m_['pop_coverage_pct'], 1, 'fr')} pour le mobile. Dans ce pays, le mobile est "
        + ("de fait la technologie d’accès." if mobile_first
           else "largement complété par l’accès fixe."))

# ── 7 · concentration de l'échantillon ─────────────────────────────────────────────
_d = latest[latest.service == MAIN_SERVICE]
conc = _d.nlargest(max(1, len(_d) // 100), "tests")["tests"].sum() / max(_d["tests"].sum(), 1) * 100
add("warn",
    "Sampling concentration",
    f"The busiest 1% of tiles carry <b>{pct(conc, 0)}</b> of all {MAIN_SERVICE} tests. Any "
    "national average is therefore dominated by a very small number of locations — a further "
    "argument for the population-weighted median.",
    "Concentration de l’échantillon",
    f"Le 1 % de carreaux les plus actifs concentrent <b>{pct(conc, 0, 'fr')}</b> de l’ensemble des "
    f"tests {SVC_FR}. Toute moyenne nationale est donc dominée par un très petit nombre de lieux — "
    "un argument de plus en faveur de la médiane pondérée par la population.")

# ── 8 · évolution ──────────────────────────────────────────────────────────────
if len(QUARTERS) > 1:
    tr = (df[df.service == MAIN_SERVICE]
          .groupby(["year", "quarter"])[IND_COLS].apply(indicators).reset_index()
          .sort_values(["year", "quarter"]))
    first, last = tr.iloc[0], tr.iloc[-1]
    chg = (last["d_median_pop"] - first["d_median_pop"]) / max(first["d_median_pop"], 1e-9) * 100
    p0 = f"{int(first['year'])} Q{int(first['quarter'])}"
    p1 = f"{int(last['year'])} Q{int(last['quarter'])}"
    add("info",
        "Trend over the observed quarters",
        f"Between {p0} and {p1}, the population-weighted median {MAIN_SERVICE} speed moved from "
        f"{mbps(first['d_median_pop'])} to {mbps(last['d_median_pop'])} "
        f"(<b>{'+' if chg >= 0 else '−'}{num(abs(chg), 0)}%</b>), while the number of measured "
        f"tiles went from {num(first['tiles'], 0)} to {num(last['tiles'], 0)}. Read the two "
        "together: more tests in new places can lower the average without any network degrading.",
        "Évolution sur les trimestres observés",
        f"Entre {p0} et {p1}, le débit médian {SVC_FR} pondéré par la population est passé de "
        f"{mbps(first['d_median_pop'], 'fr')} à {mbps(last['d_median_pop'], 'fr')} "
        f"(<b>{'+' if chg >= 0 else '−'}{pct(abs(chg), 0, 'fr')}</b>), tandis que le nombre de "
        f"carreaux mesurés passait de {num(first['tiles'], 0, 'fr')} à "
        f"{num(last['tiles'], 0, 'fr')}. Il faut lire les deux ensemble : davantage de tests dans "
        "de nouveaux lieux peut faire baisser la moyenne sans qu’aucun réseau ne se dégrade.")
else:
    tr = None

for i in INSIGHTS:
    callout(f"<b>{i[NB_LANG]['title']}.</b> {i[NB_LANG]['text']}", i["kind"])
print(f"{len(INSIGHTS)} constats générés dans {len(LANGS)} langues.")

# 13 · Graphiques

Six figures, chacune répondant à une question, et chacune **construite une fois par langue**. Le
calcul n'a lieu qu'une seule fois ; seuls les libellés changent, ce qui garantit que les versions
française et anglaise d'un graphique ne peuvent jamais afficher des chiffres différents.

Elles sont conservées dans `FIGS[langue][nom]` pour que le tableau de bord puisse les intégrer sans
rien recalculer.

Un mot sur la discipline graphique, puisque ces figures finiront dans une publication officielle :
pas d'ornement inutile, pas d'encombrement de la grille, pas de légende quand une étiquette directe
suffit, les sources sur la figure, et les prévisions signalées comme telles. La palette est fixée
par la charte institutionnelle et la couleur porte du sens — vert pour la performance, ocre pour la
prudence, brique pour le risque — elle n'est donc jamais réattribuée d'un graphique à l'autre.

In [ ]:
FIGS = {l: {} for l in LANGS}

def finish(fig, title, subtitle, lang, height=430):
    fig.update_layout(
        height=height,
        title=dict(text=f"<b>{title}</b>" +
                        (f"<br><span style='font-size:12px;color:{SLATE}'>{subtitle}</span>"
                         if subtitle else ""),
                   x=0.01, xanchor="left", y=0.94, font=dict(size=17)),
        margin=dict(l=60, r=30, t=(78 if subtitle else 60), b=70),
    )
    fig.add_annotation(text=f"<i>{T('src_note', lang, **ctx(lang))}</i>",
                       xref="paper", yref="paper", x=0, y=-0.20, showarrow=False,
                       align="left", font=dict(size=9, color=SLATE))
    return fig


def svc_label(service, lang):
    return {"en": {"fixed": "Fixed", "mobile": "Mobile"},
            "fr": {"fixed": "Fixe",  "mobile": "Mobile"}}[lang][service]


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 1 · distribution des débits mesurés
# ══════════════════════════════════════════════════════════════════════════════
def fig_distribution(lang):
    C = ctx(lang)
    fig = go.Figure()
    for i, s in enumerate(SERVICES):
        d = latest[latest.service == s]
        if d.empty:
            continue
        fig.add_trace(go.Histogram(
            x=d["d_mbps"].clip(upper=d["d_mbps"].quantile(0.99)),
            name=svc_label(s, lang), opacity=0.72, nbinsx=60,
            marker_color=[GREEN, TEAL][i % 2],
            hovertemplate="%{x:.0f} " + T("u_mbps", lang) + "<br>%{y} " +
                          T("c1_tiles", lang) + "<extra>" + svc_label(s, lang) + "</extra>"))
    fig.add_vline(x=BROADBAND_MBPS, line=dict(color=OCHRE, dash="dash", width=2),
                  annotation_text=f" {BROADBAND_MBPS} {T('u_mbps', lang)}",
                  annotation_position="top", annotation_font=dict(color=OCHRE, size=11))
    fig.update_layout(barmode="overlay", xaxis_title=T("c1_x", lang),
                      yaxis_title=T("c1_y", lang))
    return finish(fig, T("c1_title", lang, **C), T("c1_sub", lang, **C), lang)


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 2 · classement infranational
# ══════════════════════════════════════════════════════════════════════════════
_rank = main.dropna(subset=["d_median_pop"]).sort_values("d_median_pop").tail(22)
_lo, _hi = float(_rank["d_median_pop"].min()), float(_rank["d_median_pop"].max())
_pos = (np.full(len(_rank), len(RAMP) // 2) if _hi <= _lo else
        np.interp(_rank["d_median_pop"], (_lo, _hi), (0, len(RAMP) - 1)))
_RANKCOL = [RAMP[min(len(RAMP) - 1, int(v))] for v in _pos]

def fig_ranking(lang):
    C = ctx(lang)
    h = T("c2_h", lang)
    fig = go.Figure(go.Bar(
        x=_rank["d_median_pop"], y=_rank["admin_name"], orientation="h",
        marker_color=_RANKCOL, marker_line=dict(width=0),
        text=[num(v, 1, lang) for v in _rank["d_median_pop"]], textposition="outside",
        textfont=dict(size=11, color=INK),
        customdata=np.stack([_rank["pop_total"], _rank["pop_coverage_pct"], _rank["tests"]], -1),
        hovertemplate=(f"<b>%{{y}}</b><br>{h[0]}: %{{x:.1f}} {T('u_mbps', lang)}"
                       f"<br>{h[1]}: %{{customdata[0]:,.0f}}"
                       f"<br>{h[2]}: %{{customdata[1]:.1f}}%"
                       f"<br>{h[3]}: %{{customdata[2]:,.0f}}<extra></extra>")))
    fig.add_vline(x=float(national.loc[MAIN_SERVICE, "d_median_pop"]),
                  line=dict(color=INK, dash="dot", width=1.5),
                  annotation_text=" " + T("c2_nat", lang), annotation_position="top",
                  annotation_font=dict(size=10, color=INK))
    fig.update_layout(xaxis_title=T("c2_x", lang), yaxis_title="",
                      yaxis=dict(tickfont=dict(size=11)))
    return finish(fig, T("c2_title", lang, **C), T("c2_sub", lang, **C), lang,
                  height=max(430, 22 * len(_rank) + 170))


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 3 · densité et débit
# ══════════════════════════════════════════════════════════════════════════════
_sc = main.dropna(subset=["d_median_pop", "density"])
if len(_sc) > 3:
    _lx = np.log10(_sc["density"].clip(lower=1))
    _b, _a = np.polyfit(_lx, _sc["d_median_pop"], 1)
    _r = float(np.corrcoef(_lx, _sc["d_median_pop"])[0, 1])
else:
    _lx = _b = _a = _r = None

def fig_density(lang):
    C = ctx(lang)
    h = T("c3_h", lang)
    fig = go.Figure(go.Scatter(
        x=_sc["density"], y=_sc["d_median_pop"], mode="markers+text",
        text=[n if p > _sc["pop_total"].quantile(0.80) else ""
              for n, p in zip(_sc["admin_name"], _sc["pop_total"])],
        textposition="top center", textfont=dict(size=9, color=SLATE),
        marker=dict(size=np.sqrt(_sc["pop_total"]) / np.sqrt(_sc["pop_total"]).max() * 42 + 7,
                    color=_sc["pop_coverage_pct"], colorscale=[[0, "#C9D6D0"], [1, DEEP]],
                    showscale=True, line=dict(width=1, color="white"),
                    colorbar=dict(title=dict(text=T("c3_cbar", lang), font=dict(size=10)),
                                  thickness=12, len=0.65)),
        customdata=np.stack([_sc["pop_total"], _sc["pop_coverage_pct"]], -1),
        hovertemplate=(f"<b>%{{text}}</b><br>{h[0]}: %{{x:,.0f}} /km²<br>"
                       f"{h[1]}: %{{y:.1f}} {T('u_mbps', lang)}<br>"
                       f"{h[2]}: %{{customdata[0]:,.0f}}<extra></extra>")))
    if _lx is not None:
        xs = np.linspace(_lx.min(), _lx.max(), 50)
        fig.add_trace(go.Scatter(x=10 ** xs, y=_a + _b * xs, mode="lines",
                                 line=dict(color=OCHRE, dash="dash", width=2),
                                 hoverinfo="skip", showlegend=False))
        fig.add_annotation(x=0.98, y=0.06, xref="paper", yref="paper", showarrow=False,
                           text=f"<b>r = {num(_r, 2, lang)}</b> {T('c3_r', lang)}",
                           font=dict(size=12, color=OCHRE))
    fig.update_layout(xaxis=dict(title=T("c3_x", lang), type="log"),
                      yaxis_title=T("c3_y", lang))
    return finish(fig, T("c3_title", lang, **C), T("c3_sub", lang, **C), lang)


for l in LANGS:
    FIGS[l]["distribution"] = fig_distribution(l)
    FIGS[l]["ranking"] = fig_ranking(l)
    FIGS[l]["density"] = fig_density(l)

FIGS[NB_LANG]["distribution"].show()
FIGS[NB_LANG]["ranking"].show()
FIGS[NB_LANG]["density"].show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  FIG 4 · évolution trimestrielle
# ══════════════════════════════════════════════════════════════════════════════
if len(QUARTERS) > 1:
    trend = (df.groupby(["service", "year", "quarter"])[IND_COLS]
               .apply(indicators).reset_index().sort_values(["year", "quarter"]))
    trend["period"] = trend["year"].astype(str) + " Q" + trend["quarter"].astype(str)

    def fig_trend(lang):
        C = ctx(lang)
        fig = go.Figure()
        for i, s in enumerate(SERVICES):
            t = trend[trend.service == s]
            if t.empty:
                continue
            sl = svc_label(s, lang)
            fig.add_trace(go.Scatter(
                x=t["period"], y=t["d_median_pop"], name=T("c4_line", lang, service=sl),
                mode="lines+markers+text", line=dict(color=[GREEN, TEAL][i % 2], width=3),
                marker=dict(size=9), text=[num(v, 0, lang) for v in t["d_median_pop"]],
                textposition="top center", textfont=dict(size=10, color=SLATE)))
            fig.add_trace(go.Bar(x=t["period"], y=t["tiles"],
                                 name=T("c4_bar", lang, service=sl),
                                 marker_color=[MINT, "#DCEDEF"][i % 2], yaxis="y2", opacity=0.9))
        fig.update_layout(
            barmode="group", yaxis=dict(title=T("c4_y", lang)),
            yaxis2=dict(title=T("c4_y2", lang), overlaying="y", side="right", showgrid=False,
                        tickfont=dict(color=SLATE, size=10)),
            legend=dict(orientation="h", y=1.10, x=0))
        return finish(fig, T("c4_title", lang, **C), T("c4_sub", lang, **C), lang)

    for l in LANGS:
        FIGS[l]["trend"] = fig_trend(l)
    FIGS[NB_LANG]["trend"].show()
else:
    trend = None
    print("Mettez N_QUARTERS > 1 dans la cellule de configuration pour produire le graphique d'évolution.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  FIG 5 · courbe de Lorenz de la connectivité
# ══════════════════════════════════════════════════════════════════════════════
_dl = latest[latest.service == MAIN_SERVICE].dropna(subset=["d_mbps", "pop_tile"])
_dl = _dl[_dl["pop_tile"] > 0].sort_values("d_mbps")
CUM_POP = np.cumsum(_dl["pop_tile"].values) / _dl["pop_tile"].sum() * 100
_cap = np.cumsum(_dl["pop_tile"].values * _dl["d_mbps"].values)
CUM_CAP = _cap / _cap[-1] * 100
_integ = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
gini = float(1 - 2 * _integ(CUM_CAP / 100, CUM_POP / 100))

def fig_lorenz(lang):
    C = ctx(lang, gini=num(gini, 3, lang))
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode="lines", name=T("c5_eq", lang),
                             line=dict(color=SAGE, dash="dash", width=2), hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=CUM_POP, y=CUM_CAP, mode="lines", name=T("c5_obs", lang),
                             line=dict(color=DEEP, width=3), fill="tonexty",
                             fillcolor="rgba(0,168,106,0.13)",
                             hovertemplate=T("c5_h", lang) + "<extra></extra>"))
    for q in (50, 80):
        y = float(np.interp(q, CUM_POP, CUM_CAP))
        fig.add_annotation(x=q, y=y, ax=35, ay=-35, arrowhead=0, arrowcolor=OCHRE,
                           text=T("c5_a", lang, v=num(y, 0, lang), q=q),
                           font=dict(size=10, color=OCHRE), bgcolor="white",
                           bordercolor=OCHRE, borderwidth=1, borderpad=4)
    fig.update_layout(xaxis_title=T("c5_x", lang), yaxis_title=T("c5_y", lang),
                      legend=dict(orientation="h", y=1.08, x=0))
    return finish(fig, T("c5_title", lang, **C), T("c5_sub", lang, **C), lang)


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 6 · type d'habitat × service
# ══════════════════════════════════════════════════════════════════════════════
_sw = settle.dropna(subset=["d_median_pop"])

def fig_settlement(lang):
    C = ctx(lang)
    fig = go.Figure()
    for i, s in enumerate(SERVICES):
        t = _sw[_sw.service == s]
        if t.empty:
            continue
        sl = svc_label(s, lang)
        fig.add_trace(go.Bar(
            x=[T(str(v), lang) for v in t["settlement"]], y=t["d_median_pop"], name=sl,
            marker_color=[GREEN, TEAL][i % 2],
            text=[num(v, 1, lang) for v in t["d_median_pop"]], textposition="outside",
            customdata=np.stack([t["pop_total"], t["pop_coverage_pct"]], -1),
            hovertemplate=(f"<b>%{{x}}</b> — {sl}<br>%{{y:.1f}} {T('u_mbps', lang)}<br>"
                           f"{T('th_pop', lang)}: %{{customdata[0]:,.0f}}<br>"
                           f"{T('th_meas', lang)}: %{{customdata[1]:.1f}}%<extra></extra>")))
    t = _sw[_sw.service == MAIN_SERVICE]
    fig.add_trace(go.Scatter(
        x=[T(str(v), lang) for v in t["settlement"]], y=t["pop_coverage_pct"],
        name=T("c6_cov", lang), mode="lines+markers", yaxis="y2",
        line=dict(color=OCHRE, width=2.5, dash="dot"), marker=dict(size=10, symbol="diamond")))
    fig.update_layout(barmode="group", yaxis_title=T("c6_y", lang),
                      yaxis2=dict(title=T("c6_y2", lang), overlaying="y", side="right",
                                  range=[0, 100], showgrid=False,
                                  tickfont=dict(color=OCHRE, size=10)),
                      legend=dict(orientation="h", y=1.10, x=0))
    return finish(fig, T("c6_title", lang, **C), T("c6_sub", lang, **C), lang)


for l in LANGS:
    FIGS[l]["lorenz"] = fig_lorenz(l)
    FIGS[l]["settlement"] = fig_settlement(l)

FIGS[NB_LANG]["lorenz"].show()
FIGS[NB_LANG]["settlement"].show()
print(f"Gini de connectivité ({MAIN_SERVICE}, {LATEST_Y} T{LATEST_Q}) : {gini:.3f}")

# 14 · Cartes interactives

Trois couches, chacune construite une fois par langue afin que les infobulles, les légendes et les
noms de couches suivent le lecteur :

1. **Choroplèthe** — débit médian pondéré par la population, par unité administrative, avec la
   série complète d'indicateurs dans l'infobulle.
2. **Grille de mesure** — les carreaux Ookla eux-mêmes, agrégés à un niveau de quadkey plus
   grossier lorsqu'ils sont trop nombreux à dessiner ; les navigateurs cessent d'être agréables
   au-delà d'environ 15 000 polygones.
3. **Carte des lacunes** — là où des habitants vivent *sans* aucune mesure. C'est habituellement la
   couche la plus pertinente pour l'action publique, et celle que personne ne produit.

In [ ]:
CENTER = [(BBOX[1] + BBOX[3]) / 2, (BBOX[0] + BBOX[2]) / 2]
SPAN = max(BBOX[2] - BBOX[0], BBOX[3] - BBOX[1])
ZOOM0 = int(max(4, min(10, round(8.5 - math.log2(max(SPAN, 0.05)) * 0.9))))
TIP_STYLE = (f"background:white;border:1px solid {SAGE};border-radius:4px;padding:8px;"
             f"font-family:{FONT};font-size:12px;color:{INK};"
             "box-shadow:0 2px 6px rgba(0,0,0,.15)")

def base_map(zoom=None, tiles="CartoDB positron"):
    m = folium.Map(location=CENTER, zoom_start=zoom or ZOOM0, tiles=tiles,
                   control_scale=True, prefer_canvas=True)
    Fullscreen(position="topright").add_to(m)
    return m

def legend_html(title, entries, note=""):
    rows = "".join(
        f'<div style="display:flex;align-items:center;gap:7px;margin:2px 0">'
        f'<span style="width:15px;height:11px;background:{c};display:inline-block;'
        f'border:1px solid rgba(0,0,0,.18)"></span><span>{l}</span></div>'
        for c, l in entries)
    return folium.Element(f"""
    <div style="position:fixed;bottom:22px;left:14px;z-index:9999;background:white;
                border:1px solid {SAGE};border-radius:5px;padding:9px 12px;
                font-family:{FONT};font-size:11.5px;color:{INK};
                box-shadow:0 2px 6px rgba(0,0,0,.14);max-width:235px">
      <div style="font-weight:700;font-size:10px;letter-spacing:1.6px;
                  text-transform:uppercase;color:{DEEP};margin-bottom:5px">{title}</div>
      {rows}
      <div style="color:{SLATE};font-size:10px;margin-top:5px;line-height:1.4">{note}</div>
    </div>""")

MAPS = {l: {} for l in LANGS}
print(f"Centre de carte {CENTER[0]:.2f}, {CENTER[1]:.2f} · zoom initial {ZOOM0}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CARTE 1 · choroplèthe administrative
# ══════════════════════════════════════════════════════════════════════════════
amap = admin.merge(
    main[["admin_name", "d_median_pop", "d_median_tile", "pop_total", "pop_coverage_pct",
          "pct_above_bb", "tests", "tiles", "divide_ratio", "lat_median_pop", "density"]],
    on="admin_name", how="left")

_vals = amap["d_median_pop"].dropna()
_qs = (np.unique(np.quantile(_vals, np.linspace(0, 1, len(RAMP) + 1)))
       if len(_vals) >= 2 else np.array([]))

def admin_colormap(lang):
    if len(_qs) >= 3:
        c = cm.StepColormap(RAMP[:len(_qs) - 1], index=list(_qs),
                            vmin=float(_qs[0]), vmax=float(_qs[-1]))
    else:
        c = cm.LinearColormap(RAMP, vmin=0,
                              vmax=max(1, float(_vals.max() if len(_vals) else 1)))
    c.caption = T("mp_cbar1", lang, service=svc_label(MAIN_SERVICE, lang))
    return c

def build_map_admin(lang):
    C = ctx(lang, service=svc_label(MAIN_SERVICE, lang))
    cmap = admin_colormap(lang)

    def style(feat):
        v = feat["properties"].get("d_median_pop")
        ok = v is not None and np.isfinite(v)
        return {"fillColor": cmap(v) if ok else "#E4E9E6",
                "color": "white", "weight": 1.1, "fillOpacity": 0.86}

    m = base_map()
    folium.GeoJson(
        amap.to_json(), name=T("mp_layer_speed", lang, service=svc_label(MAIN_SERVICE, lang)),
        style_function=style,
        highlight_function=lambda f: {"weight": 3, "color": GOLD, "fillOpacity": 0.95},
        tooltip=folium.GeoJsonTooltip(
            fields=["admin_name", "pop_total", "pop_coverage_pct", "d_median_pop",
                    "d_median_tile", "pct_above_bb", "lat_median_pop", "tests"],
            aliases=[T("mp_t_unit", lang, level=ADMIN_LEVEL), T("mp_t_pop", lang),
                     T("mp_t_meas", lang), T("mp_t_person", lang), T("mp_t_tile", lang),
                     T("mp_t_above", lang, bb=BROADBAND_MBPS), T("mp_t_lat", lang),
                     T("mp_t_tests", lang)],
            localize=True, sticky=False, style=TIP_STYLE)).add_to(m)
    cmap.add_to(m)
    folium.GeoJson(adm0.to_json(), name=T("mp_boundary", lang),
                   style_function=lambda f: {"color": INK, "weight": 2, "fill": False}).add_to(m)
    folium.LayerControl(collapsed=True).add_to(m)
    m.get_root().html.add_child(folium.Element(
        f'<div style="position:fixed;top:12px;left:60px;z-index:9999;background:white;'
        f'border-left:4px solid {GREEN};padding:8px 14px;font-family:{FONT};'
        f'box-shadow:0 2px 6px rgba(0,0,0,.14);border-radius:0 4px 4px 0">'
        f'<div style="font-size:9.5px;letter-spacing:2px;font-weight:700;color:{DEEP}">'
        f'{COUNTRY_NAME.upper()} · {LATEST_Y} Q{LATEST_Q}</div>'
        f'<div style="font-size:14px;font-weight:700;color:{INK}">'
        f'{T("mp_m1_title", lang, **C)}</div></div>'))
    return m

for l in LANGS:
    MAPS[l]["admin"] = build_map_admin(l)
MAPS[NB_LANG]["admin"]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CARTE 2 · la grille de mesure elle-même
# ══════════════════════════════════════════════════════════════════════════════
tl = latest[latest.service == MAIN_SERVICE].copy()
agg_zoom = 16
while len(tl["quadkey"].str[:agg_zoom].unique()) > MAX_MAP_TILES and agg_zoom > 10:
    agg_zoom -= 1

if agg_zoom < 16:
    tl["qk"] = tl["quadkey"].str[:agg_zoom]
    tl["_w"] = tl["tests"].clip(lower=1)
    tl["_wd"] = tl["d_mbps"] * tl["_w"]
    g = (tl.groupby("qk").agg(_wd=("_wd", "sum"), _w=("_w", "sum"), tests=("tests", "sum"),
                              pop=("pop_tile", "sum"), n=("quadkey", "size")).reset_index())
    g["d_mbps"] = g["_wd"] / g["_w"]
    grid = pd.concat([g, quadkeys_to_bounds(g["qk"].values, zoom=agg_zoom)], axis=1)
    CELL_LABEL = {l: T("mp_agg", l, z=agg_zoom,
                       km=f"{611 * 2 ** (16 - agg_zoom) / 1000:.1f}") for l in LANGS}
else:
    grid = tl.rename(columns={"pop_tile": "pop"}).copy()
    grid["n"] = 1
    CELL_LABEL = {l: T("mp_native", l) for l in LANGS}

_gv = grid["d_mbps"].dropna()
_br = np.unique(np.quantile(_gv, np.linspace(0, 1, len(RAMP) + 1)))

GRID_FEATURES = [{
    "type": "Feature",
    "geometry": {"type": "Polygon", "coordinates": [[
        [r.west, r.south], [r.east, r.south], [r.east, r.north],
        [r.west, r.north], [r.west, r.south]]]},
    "properties": {"speed": round(float(r.d_mbps), 1), "tests": int(r.tests),
                   "pop": int(r.pop), "n": int(r.n)}} for r in grid.itertuples()]

def build_map_grid(lang):
    sl = svc_label(MAIN_SERVICE, lang)
    if len(_br) >= 3:
        tmap = cm.StepColormap(RAMP[:len(_br) - 1], index=list(_br),
                               vmin=float(_br[0]), vmax=float(_br[-1]))
    else:
        tmap = cm.LinearColormap(RAMP, vmin=float(_gv.min()), vmax=float(max(_gv.max(), 1)))
    tmap.caption = T("mp_cbar2", lang, cell=CELL_LABEL[lang])
    m = base_map(zoom=ZOOM0)
    folium.GeoJson(
        {"type": "FeatureCollection", "features": GRID_FEATURES},
        name=T("mp_layer_grid", lang, service=sl),
        style_function=lambda f: {"fillColor": tmap(f["properties"]["speed"]),
                                  "color": "none", "fillOpacity": 0.78},
        tooltip=folium.GeoJsonTooltip(
            fields=["speed", "tests", "pop", "n"],
            aliases=[T("mp_t_speed", lang), T("mp_t_tests", lang),
                     T("mp_t_estpop", lang), T("mp_t_z16", lang)],
            style=TIP_STYLE)).add_to(m)
    HeatMap([[r.lat, r.lon, float(r.tests)] for r in grid.itertuples()],
            name=T("mp_layer_heat", lang), radius=13, blur=18, min_opacity=0.25,
            gradient={0.2: "#E8F5EF", 0.5: GREEN, 0.8: DEEP, 1.0: FOREST},
            show=False).add_to(m)
    folium.GeoJson(adm0.to_json(), name=T("mp_boundary", lang),
                   style_function=lambda f: {"color": INK, "weight": 1.6, "fill": False}).add_to(m)
    tmap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

for l in LANGS:
    MAPS[l]["grid"] = build_map_grid(l)
print(f"{len(GRID_FEATURES):,} polygones dessinés · {CELL_LABEL[NB_LANG]}")
MAPS[NB_LANG]["grid"]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CARTE 3 · la lacune de mesure — là où des habitants vivent sans aucun test
# ══════════════════════════════════════════════════════════════════════════════
gap_cells = cells[(~cells["measured_any"]) & (cells["pop"] > 0)]
meas_cells = cells[cells["measured_any"] & (cells["pop"] > 0)]
GAP_POP = float(gap_cells["pop"].sum())
_gap_top = gap_cells.nlargest(min(6000, len(gap_cells)), "pop")
_meas_top = meas_cells.nlargest(min(6000, len(meas_cells)), "pop")

def build_map_gap(lang):
    m = base_map(tiles="CartoDB dark_matter")
    folium.GeoJson(adm0.to_json(), name=T("mp_boundary", lang),
                   style_function=lambda f: {"color": "#FFFFFF", "weight": 1.4,
                                             "fill": False, "opacity": 0.6}).add_to(m)
    HeatMap([[r.lat, r.lon, float(r.pop)] for r in _gap_top.itertuples()],
            name=T("mp_layer_gap", lang), radius=12, blur=16, min_opacity=0.32,
            gradient={0.2: "#F6D58A", 0.5: OCHRE, 0.8: TERRA, 1.0: BRICK}).add_to(m)
    HeatMap([[r.lat, r.lon, float(r.pop)] for r in _meas_top.itertuples()],
            name=T("mp_layer_meas", lang), radius=11, blur=15, min_opacity=0.28,
            gradient={0.2: "#9ED9C0", 0.6: GREEN, 1.0: "#00FFB0"}, show=False).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.get_root().html.add_child(legend_html(
        T("mp_m3_legend", lang),
        [(BRICK, T("mp_m3_a", lang)), (OCHRE, T("mp_m3_b", lang)), (GREEN, T("mp_m3_c", lang))],
        T("mp_m3_note", lang, pop=big(GAP_POP, lang), period=f"{LATEST_Y} Q{LATEST_Q}")))
    return m

for l in LANGS:
    MAPS[l]["gap"] = build_map_gap(l)

kpi_row([("Population sans aucune mesure", big(GAP_POP, "fr"), "", BRICK),
         ("Part de la population nationale", f"{100 * GAP_POP / POP_TOTAL:.1f}".replace(".", ","), "%", OCHRE),
         ("Cellules peuplées non mesurées", f"{len(gap_cells):,}", "", SLATE)])
callout("Lisez cette carte comme une carte d'<b>échantillonnage</b>, pas de couverture. Elle "
        "montre où Ookla n'a aucune observation — ce qui peut signifier aucun réseau, aucun "
        "smartphone, aucun forfait de données, ou simplement aucune raison de lancer un test. "
        "Distinguer ces quatre cas exige des données d'enquête ménages ou d'opérateurs, et cette "
        "distinction est la limite honnête de ce produit.", "risk")
MAPS[NB_LANG]["gap"]

# 15 · Le tableau de bord

Tout ce qui précède est assemblé dans **un seul fichier HTML autonome** portant les deux versions
linguistiques, avec un sélecteur dans le coin supérieur droit. Aucune étape de compilation, aucun
serveur, aucun framework : ouvrez-le localement, envoyez-le par courriel, ou poussez-le dans un
dépôt et il devient une URL publique. Cette propriété compte plus qu'il n'y paraît — c'est elle qui
rend le produit viable après la formation, quand il ne reste plus personne pour maintenir une
application Node.

**Comment fonctionne le sélecteur.** Les deux versions sont écrites dans le même document, à
l'intérieur de `<div class="lang-block" data-lang="…">`. Une seule règle CSS affiche l'une et masque
l'autre, et quatre lignes de JavaScript inversent l'attribut, mémorisent le choix dans le navigateur
et demandent à Plotly de redimensionner les graphiques qui viennent d'apparaître. La page s'ouvre
d'elle-même en français pour un lecteur dont le navigateur est configuré en français. Aucun
aller-retour vers un serveur, aucune dépendance à un service de traduction — le fichier fonctionne
hors ligne, depuis une clé USB si besoin.

**Ce qui y entre :** la bande d'indicateurs clés, les constats automatiques, les trois cartes
interactives, les six graphiques, le tableau infranational, et un bloc méthodologique avec les
licences et la déclaration des limites — le tout en double.

**Ce qui n'y entre pas :** tout ce qui exigerait une clé, une authentification ou un service payant.

In [ ]:
from string import Template
import html as _html

def fig_html(fig, div_id):
    return pio.to_html(fig, full_html=False, include_plotlyjs=False, div_id=div_id,
                       config={"displayModeBar": False, "responsive": True})

def map_html(m, height=560):
    """Intègre une carte folium dans un iframe autonome et responsive."""
    raw = m.get_root().render()
    return (f'<iframe srcdoc="{_html.escape(raw, quote=True)}" loading="lazy" '
            f'style="width:100%;height:{height}px;border:1px solid {SAGE};'
            f'border-radius:6px;background:white"></iframe>')


# ══════════════════════════════════════════════════════════════════════════════
#  Blocs de contenu, un par langue
# ══════════════════════════════════════════════════════════════════════════════
nat = national.loc[MAIN_SERVICE]

def kpi_block(lang):
    items = [
        (T("kpi_pop", lang),        big(POP_TOTAL, lang),
         T("u_wp", lang, y=WORLDPOP_YEAR), GREEN),
        (T("kpi_med_person", lang), num(nat["d_median_pop"], 1, lang), T("u_mbps", lang), DEEP),
        (T("kpi_med_tile", lang),   num(nat["d_median_tile"], 1, lang), T("u_mbps", lang), SLATE),
        (T("kpi_above", lang, bb=BROADBAND_MBPS), num(nat["pct_above_bb"], 0, lang),
         T("u_pct_meas", lang), TEAL),
        (T("kpi_pop_meas", lang),   num(nat["pop_coverage_pct"], 0, lang), T("u_pct", lang), OCHRE),
        (T("kpi_area_meas", lang),  num(nat["area_coverage_pct"], 2, lang), T("u_pct", lang), TERRA),
        (T("kpi_gini", lang),       num(gini, 2, lang), "0–1", BRICK),
        (T("kpi_lat", lang),        num(nat["lat_median_pop"], 0, lang), T("u_ms", lang), INK),
    ]
    return "".join(f"""
      <div class="kpi"><div class="kpi-l">{lab}</div>
        <div class="kpi-v" style="color:{col}">{val}<span class="kpi-u">{unit}</span></div>
      </div>""" for lab, val, unit, col in items)


ICOL = {"info": GREEN, "warn": OCHRE, "risk": BRICK}
IBG = {"info": MINT, "warn": "#FDF4E0", "risk": "#FBECEA"}

def insights_block(lang):
    return "".join(f"""
      <div class="ins" style="background:{IBG[i['kind']]};border-color:{ICOL[i['kind']]}">
        <div class="ins-t" style="color:{ICOL[i['kind']]}">{i[lang]['title']}</div>
        <div class="ins-b">{i[lang]['text']}</div>
      </div>""" for i in INSIGHTS)


def table_block(lang):
    rows = []
    for r in main.itertuples():
        if not np.isfinite(r.d_median_pop):
            continue
        rows.append(
            "<tr>"
            f"<td class='nm'>{r.admin_name}</td>"
            f"<td>{num(r.pop_total, 0, lang)}</td>"
            f"<td>{pct(r.pop_coverage_pct, 1, lang)}</td>"
            f"<td>{num(r.tests, 0, lang)}</td>"
            f"<td class='hi'>{num(r.d_median_pop, 1, lang)}</td>"
            f"<td>{num(r.d_median_tile, 1, lang)}</td>"
            f"<td>{pct(r.pct_above_bb, 0, lang)}</td>"
            f"<td>{num(r.divide_ratio, 1, lang)}×</td>"
            f"<td>{num(r.lat_median_pop, 0, lang)}</td></tr>")
    return "".join(rows)


def limitations(lang):
    unmeasured = 100 - nat["pop_coverage_pct"]
    if lang == "en":
        return (
            "<b>1 · Speedtest data is crowdsourced and self-selected.</b> A tile exists because "
            "somebody chose to run a test there — typically because they suspected a problem or had "
            "just changed connection. It is not a probability sample and carries no design weights."
            f"<br><br><b>2 · Absence of measurement is not absence of service.</b> "
            f"{pct(unmeasured, 0)} of the population lives in a cell with no test this quarter; "
            "the map of gaps is a map of Speedtest users, not of network coverage."
            "<br><br><b>3 · Device and tariff effects are not separable.</b> A slow measurement may "
            "reflect an old handset, an exhausted data bundle or a congested cell, not the network."
            f"<br><br><b>4 · Only {pct(nat['area_coverage_pct'], 2)} of the land area is "
            "measured</b>, and the busiest 1% of tiles carry a large share of all tests, so "
            "unweighted national averages are dominated by a few urban locations."
            "<br><br><b>5 · Population figures are modelled.</b> WorldPop redistributes census or "
            "projected counts using covariates including built-up area and night-time lights; it is "
            "not a census and should not be used to validate another light-derived indicator."
            "<br><br><b>6 · Boundaries are from geoBoundaries</b>, not from the national mapping "
            "authority; small differences in geometry will move the subnational figures."
            "<br><br><b>7 · Licence.</b> The Ookla licence is non-commercial and share-alike. This "
            "product and any derivative must carry CC BY-NC-SA 4.0 and may not be sold or embedded "
            "in a commercial service. Confirm with your legal service before an official release.")
    return (
        "<b>1 · Les données Speedtest sont produites par les utilisateurs et auto-sélectionnées.</b> "
        "Un carreau existe parce que quelqu’un a choisi d’y lancer un test — le plus souvent parce "
        "qu’il soupçonnait un problème ou venait de changer de connexion. Ce n’est pas un "
        "échantillon probabiliste et il ne comporte aucune pondération de sondage."
        f"<br><br><b>2 · L’absence de mesure n’est pas l’absence de service.</b> "
        f"{pct(unmeasured, 0, 'fr')} de la population vit dans une cellule sans aucun test ce "
        "trimestre ; la carte des lacunes est une carte des utilisateurs de Speedtest, pas de la "
        "couverture réseau."
        "<br><br><b>3 · Les effets du terminal et du forfait ne sont pas séparables.</b> Une mesure "
        "faible peut refléter un téléphone ancien, un forfait épuisé ou une cellule congestionnée, "
        "et non le réseau disponible."
        f"<br><br><b>4 · Seuls {pct(nat['area_coverage_pct'], 2, 'fr')} du territoire sont "
        "mesurés</b>, et le 1 % de carreaux les plus actifs concentrent une large part des tests : "
        "les moyennes nationales non pondérées sont dominées par quelques lieux urbains."
        "<br><br><b>5 · Les chiffres de population sont modélisés.</b> WorldPop redistribue des "
        "effectifs censitaires ou projetés à l’aide de covariables incluant le bâti et les lumières "
        "nocturnes ; ce n’est pas un recensement et il ne doit pas servir à valider un autre "
        "indicateur dérivé des lumières nocturnes."
        "<br><br><b>6 · Les limites administratives proviennent de geoBoundaries</b>, et non de "
        "l’autorité cartographique nationale ; de légères différences de géométrie déplacent les "
        "valeurs infranationales."
        "<br><br><b>7 · Licence.</b> La licence Ookla est non commerciale et à partage dans les "
        "mêmes conditions. Ce produit et toute œuvre dérivée doivent porter la licence "
        "CC BY-NC-SA 4.0 et ne peuvent être vendus ni intégrés à un service commercial. Faites "
        "confirmer ce point par votre service juridique avant toute diffusion officielle.")

print("Constructeurs de contenu prêts.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Feuille de style (palette institutionnelle inspirée de la BAD)
# ══════════════════════════════════════════════════════════════════════════════
CSS = Template(r"""
  :root{--green:$GREEN;--deep:$DEEP;--forest:$FOREST;--gold:$GOLD;--ochre:$OCHRE;
        --ink:$INK;--slate:$SLATE;--mist:$MIST;--mint:$MINT;--sage:$SAGE;--brick:$BRICK;}
  *{box-sizing:border-box}
  body{margin:0;background:#fff;color:var(--ink);
       font-family:Calibri,'Segoe UI',Helvetica,Arial,sans-serif;line-height:1.6}
  .wrap{max-width:1220px;margin:0 auto;padding:0 26px}
  .lang-block{display:none}
  html[data-lang="en"] .lang-block[data-lang="en"]{display:block}
  html[data-lang="fr"] .lang-block[data-lang="fr"]{display:block}
  #langbar{position:fixed;top:14px;right:18px;z-index:200;display:flex;gap:0;
           background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.45);
           border-radius:20px;overflow:hidden;backdrop-filter:blur(4px)}
  #langbar button{border:0;background:transparent;color:#fff;font-weight:700;font-size:12px;
        letter-spacing:1.4px;padding:7px 15px;cursor:pointer;font-family:inherit}
  #langbar button.on{background:var(--gold);color:var(--ink)}
  header{background:linear-gradient(120deg,var(--forest) 0%,var(--deep) 45%,var(--green) 100%);
         color:#fff;padding:52px 0 44px;position:relative;overflow:hidden}
  header:after{content:"";position:absolute;right:-90px;top:-90px;width:330px;height:330px;
       border-radius:50%;border:42px solid rgba(255,255,255,.07)}
  .kick{font-size:11px;font-weight:700;letter-spacing:3.2px;text-transform:uppercase;
        color:var(--gold)}
  h1{font-size:40px;margin:11px 0 6px;font-weight:700;line-height:1.1}
  .sub{font-size:17px;font-style:italic;color:#E6F6EE;max-width:780px}
  .goldbar{height:5px;background:var(--gold);width:118px;margin-top:22px}
  nav{position:sticky;top:0;z-index:50;background:#fff;border-bottom:1px solid var(--sage);
      box-shadow:0 1px 4px rgba(0,0,0,.05)}
  nav .wrap{display:flex;gap:26px;overflow-x:auto;padding-top:13px;padding-bottom:13px}
  nav a{color:var(--slate);text-decoration:none;font-size:12.5px;font-weight:600;
        letter-spacing:.4px;white-space:nowrap;padding-bottom:2px;border-bottom:2px solid transparent}
  nav a:hover{color:var(--deep);border-bottom-color:var(--green)}
  section{padding:44px 0 8px}
  h2{font-size:26px;margin:0 0 6px}
  .h2k{font-size:10.5px;font-weight:700;letter-spacing:3px;text-transform:uppercase;
       color:var(--deep)}
  .lede{color:var(--slate);font-size:14px;max-width:880px;margin:0 0 22px}
  .kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(178px,1fr));gap:13px;
        margin:8px 0 6px}
  .kpi{background:#fff;border:1px solid var(--sage);border-radius:6px;padding:15px 17px}
  .kpi-l{font-size:9.5px;font-weight:700;letter-spacing:1.9px;text-transform:uppercase;
         color:var(--slate);line-height:1.35;min-height:26px}
  .kpi-v{font-size:33px;font-weight:700;line-height:1.14;margin-top:7px}
  .kpi-u{font-size:12.5px;font-weight:600;color:var(--slate);margin-left:5px}
  .ins-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:14px}
  .ins{border:1px solid;border-left-width:4px;border-radius:5px;padding:14px 17px}
  .ins-t{font-size:13.5px;font-weight:700;margin-bottom:5px}
  .ins-b{font-size:13px;color:var(--ink)}
  .card{background:#fff;border:1px solid var(--sage);border-radius:7px;padding:8px 10px;margin:16px 0}
  .grid2{display:grid;grid-template-columns:1fr 1fr;gap:18px}
  @media(max-width:900px){.grid2{grid-template-columns:1fr}h1{font-size:30px}}
  table{width:100%;border-collapse:collapse;font-size:12.5px}
  th{background:var(--deep);color:#fff;text-align:right;padding:9px 11px;font-weight:600;
     position:sticky;top:0;font-size:11.5px}
  th:first-child{text-align:left}
  td{padding:7px 11px;border-bottom:1px solid var(--sage);text-align:right}
  td.nm{text-align:left;font-weight:600}
  td.hi{color:var(--deep);font-weight:700}
  tbody tr:hover{background:var(--mint)}
  .tblwrap{max-height:520px;overflow:auto;border:1px solid var(--sage);border-radius:7px}
  .meth{background:var(--mist);border-radius:7px;padding:24px 28px;font-size:13.5px}
  .meth h3{margin:20px 0 7px;font-size:15px;color:var(--deep)}
  .meth h3:first-child{margin-top:0}
  .meth ul{margin:6px 0;padding-left:20px}
  .meth li{margin:4px 0}
  .warnbox{background:#FBECEA;border:1px solid var(--brick);border-left-width:4px;
           border-radius:5px;padding:15px 18px;font-size:13.5px;margin:16px 0}
  footer{background:var(--ink);color:#C9D2CD;font-size:12px;padding:32px 0;margin-top:52px}
  footer a{color:var(--gold);text-decoration:none}
  .src{font-size:10.5px;font-style:italic;color:var(--slate);margin-top:4px}
  code{background:var(--mint);padding:1px 5px;border-radius:3px;font-size:12px}
""").substitute(GREEN=GREEN, DEEP=DEEP, FOREST=FOREST, GOLD=GOLD, OCHRE=OCHRE,
                INK=INK, SLATE=SLATE, MIST=MIST, MINT=MINT, SAGE=SAGE, BRICK=BRICK)

LANG_JS = """
  function setLang(l){
    document.documentElement.setAttribute('data-lang', l);
    document.documentElement.setAttribute('lang', l);
    try{ localStorage.setItem('dashLang', l); }catch(e){}
    document.querySelectorAll('#langbar button').forEach(function(b){
      b.classList.toggle('on', b.dataset.lang === l); });
    setTimeout(function(){
      document.querySelectorAll('.lang-block[data-lang="'+l+'"] .js-plotly-plot')
        .forEach(function(p){ try{ Plotly.Plots.resize(p); }catch(e){} });
    }, 60);
  }
  (function(){
    var saved = null;
    try{ saved = localStorage.getItem('dashLang'); }catch(e){}
    var auto = (navigator.language || 'en').toLowerCase().indexOf('fr') === 0 ? 'fr' : null;
    setLang(saved || auto || DEFAULT_LANG);
  })();
"""
print(f"Feuille de style : {len(CSS):,} caractères")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Assembler un document unique contenant toutes les langues
# ══════════════════════════════════════════════════════════════════════════════
def language_block(lang):
    C = ctx(lang)
    charts = {k: fig_html(FIGS[lang][k], f"fig-{k}-{lang}") for k in FIGS[lang]}
    trend_card = (f'<div class="card">{charts["trend"]}</div>' if "trend" in charts else "")
    th = [T("th_unit", lang, level=ADMIN_LEVEL), T("th_pop", lang), T("th_meas", lang),
          T("th_tests", lang), T("th_med_person", lang), T("th_med_tile", lang),
          T("th_above", lang, bb=BROADBAND_MBPS), T("th_ratio", lang), T("th_lat", lang)]
    return f"""
<div class="lang-block" data-lang="{lang}">

<header><div class="wrap">
  <div class="kick">{T("kicker", lang, **C)}</div>
  <h1>{T("title", lang, **C)}</h1>
  <div class="sub">{T("subtitle", lang, **C)}</div>
  <div class="goldbar"></div>
</div></header>

<nav><div class="wrap">
  <a href="#kpi-{lang}">{T("nav_kpi", lang)}</a>
  <a href="#ins-{lang}">{T("nav_ins", lang)}</a>
  <a href="#maps-{lang}">{T("nav_maps", lang)}</a>
  <a href="#charts-{lang}">{T("nav_charts", lang)}</a>
  <a href="#table-{lang}">{T("nav_table", lang)}</a>
  <a href="#method-{lang}">{T("nav_method", lang)}</a>
</div></nav>

<section id="kpi-{lang}"><div class="wrap">
  <div class="h2k">{T("s1_kick", lang)}</div><h2>{T("s1_title", lang, **C)}</h2>
  <p class="lede">{T("s1_lede", lang, tests=big(nat['tests'], lang),
                     tiles=num(nat['tiles'], 0, lang), pop=big(POP_TOTAL, lang),
                     service=svc_label(MAIN_SERVICE, lang).lower())}</p>
  <div class="kpis">{kpi_block(lang)}</div>
  <div class="src">{T("s1_src", lang, **C)}</div>
</div></section>

<section id="ins-{lang}"><div class="wrap">
  <div class="h2k">{T("s2_kick", lang)}</div><h2>{T("s2_title", lang)}</h2>
  <p class="lede">{T("s2_lede", lang)}</p>
  <div class="ins-grid">{insights_block(lang)}</div>
</div></section>

<section id="maps-{lang}"><div class="wrap">
  <div class="h2k">{T("s3_kick", lang)}</div><h2>{T("s3_title", lang)}</h2>
  <p class="lede">{T("s3_lede", lang)}</p>
  <div class="card">{map_html(MAPS[lang]["admin"])}</div>
  <div class="src">{T("map1_cap", lang, **C)}</div>
  <div class="card">{map_html(MAPS[lang]["grid"])}</div>
  <div class="src">{T("map2_cap", lang)}</div>
  <div class="card">{map_html(MAPS[lang]["gap"], 520)}</div>
  <div class="src">{T("map3_cap", lang)}</div>
</div></section>

<section id="charts-{lang}"><div class="wrap">
  <div class="h2k">{T("s4_kick", lang)}</div><h2>{T("s4_title", lang)}</h2>
  <p class="lede">{T("s4_lede", lang)}</p>
  <div class="card">{charts["distribution"]}</div>
  <div class="card">{charts["ranking"]}</div>
  <div class="grid2">
    <div class="card">{charts["density"]}</div>
    <div class="card">{charts["lorenz"]}</div>
  </div>
  <div class="card">{charts["settlement"]}</div>
  {trend_card}
</div></section>

<section id="table-{lang}"><div class="wrap">
  <div class="h2k">{T("s5_kick", lang)}</div><h2>{T("s5_title", lang, **C)}</h2>
  <p class="lede">{T("s5_lede", lang)}</p>
  <div class="tblwrap"><table>
    <thead><tr>{"".join(f"<th>{h}</th>" for h in th)}</tr></thead>
    <tbody>{table_block(lang)}</tbody>
  </table></div>
</div></section>

<section id="method-{lang}"><div class="wrap">
  <div class="h2k">{T("s6_kick", lang)}</div><h2>{T("s6_title", lang)}</h2>
  <div class="meth">
    <h3>{T("h_sources", lang)}</h3>
    <ul>
      <li>{T("src_ookla", lang, **C)}</li>
      <li>{T("src_wp", lang, **C)}</li>
      <li>{T("src_gb", lang, **C)}</li>
    </ul>
    <h3>{T("h_method", lang)}</h3>
    <ul>
      <li>{T("meth_1", lang)}</li><li>{T("meth_2", lang)}</li><li>{T("meth_3", lang)}</li>
      <li>{T("meth_4", lang, **C)}</li><li>{T("meth_5", lang)}</li>
    </ul>
    <h3>{T("h_limits", lang)}</h3>
    <div class="warnbox">{limitations(lang)}</div>
    <h3>{T("h_repro", lang)}</h3>
    <ul>
      <li>{T("repro_1", lang, date=dt.date.today().isoformat(), **C)}</li>
      <li>{T("repro_2", lang)}</li>
    </ul>
  </div>
</div></section>

<footer><div class="wrap">{T("footer", lang, **C)}</div></footer>
</div>"""


buttons = "".join(
    f'<button data-lang="{l}" onclick="setLang(\'{l}\')">{l.upper()}</button>' for l in LANGS)

html_out = f"""<!DOCTYPE html>
<html lang="{LANGS[0]}" data-lang="{LANGS[0]}"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>{T("title", LANGS[0], **ctx(LANGS[0]))}</title>
<meta name="description" content="{T("meta_desc", LANGS[0], **ctx(LANGS[0]))}">
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js" charset="utf-8"></script>
<style>{CSS}</style></head><body>
<div id="langbar">{buttons}</div>
{"".join(language_block(l) for l in LANGS)}
<script>const DEFAULT_LANG = "{LANGS[0]}";{LANG_JS}</script>
</body></html>"""

DASH_PATH = Path(OUTPUT_DIR) / "index.html"
DASH_PATH.write_text(html_out, encoding="utf-8")
size_mb = DASH_PATH.stat().st_size / 1e6

banner("TABLEAU DE BORD CONSTRUIT", f"{DASH_PATH}  ·  {size_mb:.1f} Mo  ·  "
       f"{' / '.join(TXT[l]['lang_name'] for l in LANGS)}",
       "Fichier unique et autonome portant toutes les versions linguistiques. Le lecteur bascule "
       "avec les boutons en haut à droite ; le choix est mémorisé, et la page s'ouvre d'elle-même "
       "en français pour un navigateur configuré en français. Le nom <code>index.html</code> est "
       "délibéré — un serveur web le renvoie à la racine d'un site.")
if size_mb > 45:
    callout(f"Le tableau de bord pèse {size_mb:.0f} Mo, ce qui est lourd pour une page publique. "
            "Baissez <code>MAX_MAP_TILES</code> dans la cellule de configuration et "
            "reconstruisez, ou retirez une langue de <code>LANGS</code>.", "warn")

In [ ]:
# Aperçu dans le notebook (faites défiler dans le cadre, et essayez les boutons FR/EN)
display(HTML(f"""
<div style="border:1px solid {SAGE};border-radius:7px;overflow:hidden;margin-top:8px">
  <iframe src="{DASH_PATH.as_posix()}" style="width:100%;height:780px;border:0"></iframe>
</div>
<div style="font-family:{FONT};font-size:11.5px;color:{SLATE};margin-top:5px">
  Si le cadre reste vide — une restriction du bac à sable Colab/Kaggle — téléchargez
  <code>{DASH_PATH}</code> et ouvrez-le localement. Le fichier, lui, est complet.
</div>"""))

# 16 · Exports

Le tableau de bord est le produit de *communication*. Les fichiers ci-dessous sont le produit
*statistique* — ce qu'un INS pair, un chercheur ou vos propres analystes réutiliseront réellement.
La dernière cellule de cette section écrit le `README.md` bilingue et la `LICENSE` qui les
accompagneront une fois le dossier publié.

In [ ]:
stamp = f"{COUNTRY_ISO3}_{LATEST_Y}Q{LATEST_Q}"
out = Path(OUTPUT_DIR)
written = []

# 1 · indicateurs infranationaux (le livrable principal) ----------------------
export_cols = ["service", "admin_name", "pop_total", "pop_covered", "pop_coverage_pct",
               "area_coverage_pct", "tiles", "tests", "devices", "d_median_pop", "d_mean_pop",
               "d_median_tile", "d_p10_pop", "d_p90_pop", "divide_ratio", "u_median_pop",
               "lat_median_pop", "pct_above_bb", "pct_above_hi", "density"]
sub_out = sub[[c for c in export_cols if c in sub.columns]].copy()
sub_out.insert(0, "iso3", COUNTRY_ISO3)
sub_out.insert(1, "period", f"{LATEST_Y}Q{LATEST_Q}")
sub_out.insert(2, "admin_level", ADMIN_LEVEL)
p = out / f"connectivity_{ADMIN_LEVEL}_{stamp}.csv"; sub_out.to_csv(p, index=False); written.append(p)

# 2 · les mêmes, en GeoJSON, pour QGIS ou une carte web ------------------------------
p = out / f"connectivity_{ADMIN_LEVEL}_{stamp}.geojson"
amap.to_file(p, driver="GeoJSON"); written.append(p)

# 3 · synthèse nationale --------------------------------------------------------
nat_out = national.reset_index()
nat_out.insert(0, "iso3", COUNTRY_ISO3); nat_out.insert(1, "period", f"{LATEST_Y}Q{LATEST_Q}")
p = out / f"national_summary_{stamp}.csv"; nat_out.to_csv(p, index=False); written.append(p)

# 4 · fichier détail au carreau (parquet : 5 à 10 fois plus compact qu'un CSV) ----------------
tile_cols = ["quadkey", "service", "year", "quarter", "lon", "lat", "d_mbps", "u_mbps",
             "latency_ms", "tests", "devices", "pop_tile", "settlement", "admin_name"]
p = out / f"tiles_{stamp}.parquet"
df[[c for c in tile_cols if c in df.columns]].to_parquet(p, index=False); written.append(p)

# 5 · habitat + évolution ------------------------------------------------------
p = out / f"settlement_{stamp}.csv"; settle.to_csv(p, index=False); written.append(p)
if trend is not None:
    p = out / f"trend_{COUNTRY_ISO3}.csv"; trend.to_csv(p, index=False); written.append(p)

# 6 · métadonnées lisibles par machine ----------------------------------------------
meta = {
    "title": {l: T("title", l, **ctx(l)) for l in LANGS},
    "iso3": COUNTRY_ISO3, "country": COUNTRY_NAME, "languages": LANGS,
    "reference_period": f"{LATEST_Y}Q{LATEST_Q}",
    "quarters_processed": [f"{y}Q{q}" for y, q in QUARTERS],
    "services": SERVICES, "admin_level": ADMIN_LEVEL,
    "generated": dt.datetime.now().isoformat(timespec="seconds"),
    "sources": {
        "ookla": {"name": "Ookla Speedtest Open Data", "licence": "CC BY-NC-SA 4.0",
                  "url": "https://github.com/teamookla/ookla-open-data",
                  "resolution": "Web Mercator z16 (~611 m at the equator)"},
        "worldpop": {"name": f"WorldPop {WORLDPOP_YEAR} {WORLDPOP_RES} UN-adjusted",
                     "licence": "CC BY 4.0", "url": "https://www.worldpop.org"},
        "geoboundaries": {"name": "geoBoundaries gbOpen", "licence": "CC BY 4.0",
                          "url": "https://www.geoboundaries.org"}},
    "methods": {"weighting": "population-weighted median; cell population shared equally "
                             "among the tiles whose centroid falls in the cell",
                "coverage": "share of population in 1 km cells containing >= 1 measured tile",
                "thresholds_mbps": {"broadband": BROADBAND_MBPS, "high_quality": GOOD_SPEED_MBPS},
                "settlement_density_thresholds": {"urban": URBAN_DENS_MIN,
                                                  "periurban": PERIURBAN_DENS_MIN}},
    "headline": {k: (float(nat[k]) if np.isfinite(nat[k]) else None)
                 for k in ["d_median_pop", "d_median_tile", "pct_above_bb", "pct_above_hi",
                           "pop_coverage_pct", "area_coverage_pct", "lat_median_pop"]},
    "population_total": POP_TOTAL, "connectivity_gini": float(gini),
    "licence_of_this_product": "CC BY-NC-SA 4.0 (inherited from Ookla, share-alike)",
}
p = out / "metadata.json"; p.write_text(json.dumps(meta, indent=2, ensure_ascii=False),
                                        encoding="utf-8"); written.append(p)

for f in written:
    print(f"  {f.stat().st_size/1024:9,.0f} KB   {f.name}")
print(f"\n{len(written) + 1} fichiers dans {out.resolve()} (tableau de bord compris)")

In [ ]:
README = f"""# {COUNTRY_NAME} — Connectivity and Population · Connectivité et population

**Population-weighted connectivity indicators from Ookla® Speedtest Open Data and WorldPop**
**Indicateurs de connectivité pondérés par la population, à partir des données ouvertes Ookla® Speedtest et de WorldPop**

[![Dashboard](https://img.shields.io/badge/dashboard-live-00A86A)](./index.html)
![Licence](https://img.shields.io/badge/licence-CC%20BY--NC--SA%204.0-D49A00)
![Languages](https://img.shields.io/badge/langues-EN%20%7C%20FR-00704A)

African Development Bank · AU STATAFRIC — STG17.
Generated on {dt.date.today().isoformat()} for **{COUNTRY_NAME} ({COUNTRY_ISO3})**,
reference period **{LATEST_Y} Q{LATEST_Q}**. The dashboard is bilingual: use the EN / FR switch in
the top-right corner. *Le tableau de bord est bilingue : utilisez le sélecteur EN / FR en haut à droite.*

## Headline figures · Chiffres clés

| Indicator · Indicateur | Value · Valeur |
|---|---|
| Population ({WORLDPOP_YEAR}, WorldPop) | {POP_TOTAL:,.0f} |
| Median {MAIN_SERVICE} download **per person** · Débit médian **par habitant** | **{nat['d_median_pop']:.1f} Mbps** |
| Median download per measured tile · Débit médian par carreau mesuré | {nat['d_median_tile']:.1f} Mbps |
| Population ≥ {BROADBAND_MBPS} Mbps · Population ≥ {BROADBAND_MBPS} Mbit/s | {nat['pct_above_bb']:.1f}% |
| Population measured · Population mesurée | {nat['pop_coverage_pct']:.1f}% |
| Land area measured · Territoire mesuré | {nat['area_coverage_pct']:.2f}% |
| Connectivity Gini · Gini de connectivité | {gini:.3f} |
| Median latency · Latence médiane | {nat['lat_median_pop']:.0f} ms |
| {ADMIN_LEVEL} units · Unités {ADMIN_LEVEL} | {len(main)} |

## Contents · Contenu

| File | Description |
|---|---|
| `index.html` | Bilingual interactive dashboard · Tableau de bord interactif bilingue |
| `connectivity_{ADMIN_LEVEL}_{stamp}.csv` | Indicators by administrative unit · Indicateurs par unité administrative |
| `connectivity_{ADMIN_LEVEL}_{stamp}.geojson` | Same, with geometry · Idem, avec géométrie |
| `national_summary_{stamp}.csv` | National aggregates · Agrégats nationaux |
| `tiles_{stamp}.parquet` | Tile-level micro-file · Fichier détail au carreau |
| `settlement_{stamp}.csv` | Urban / peri-urban / rural · Urbain / périurbain / rural |
| `metadata.json` | Machine-readable provenance · Provenance lisible par machine |
| `.nojekyll` | Tells GitHub Pages to serve the files as they are |

## Method · Méthode

**EN.** Ookla publishes quarterly performance tiles at Web-Mercator zoom 16 (≈611 m at the equator).
Tiles covering {COUNTRY_NAME} were extracted directly from the global parquet files using a quadkey
range predicate, clipped to the national polygon on the tile centroid, and converted from kbps to
Mbps. Each tile was located in the WorldPop {WORLDPOP_YEAR} {WORLDPOP_RES} UN-adjusted population
grid by its centroid; the population of each grid cell was shared equally among the tiles it
contains, giving every tile a population weight. All headline speeds are **population-weighted
medians**. Coverage of measurement is the share of the national population living in a grid cell
that contains at least one measured tile. Settlement classes are a density proxy
(urban ≥ {URBAN_DENS_MIN} people/km², peri-urban ≥ {PERIURBAN_DENS_MIN}, rural below).

**FR.** Ookla publie des carreaux de performance trimestriels au zoom 16 en projection Web-Mercator
(≈611 m à l'équateur). Les carreaux couvrant {COUNTRY_NAME} ont été extraits directement des
fichiers parquet mondiaux au moyen d'un prédicat d'intervalle sur le quadkey, découpés sur le
polygone national selon le centroïde du carreau, puis convertis de kbit/s en Mbit/s. Chaque carreau
a été localisé dans la grille de population WorldPop {WORLDPOP_YEAR} {WORLDPOP_RES} ajustée aux
estimations des Nations unies ; la population de chaque cellule a été répartie à parts égales entre
les carreaux qu'elle contient, ce qui donne à chaque carreau un poids de population. Tous les débits
mis en avant sont des **médianes pondérées par la population**. La couverture de la mesure est la
part de la population nationale vivant dans une cellule contenant au moins un carreau mesuré. Les
classes d'habitat sont une approximation par la densité (urbain ≥ {URBAN_DENS_MIN} hab./km²,
périurbain ≥ {PERIURBAN_DENS_MIN}, rural en dessous).

## Limitations · Limites

**EN.** (1) Speedtest measurements are user-initiated and self-selected: no probability sample, no
design weights. (2) Absence of measurement is not absence of service —
{100 - nat['pop_coverage_pct']:.0f}% of the population lives in a cell with no test in the reference
quarter. (3) Device and tariff effects cannot be separated from network performance. (4) Only
{nat['area_coverage_pct']:.2f}% of the land area is measured and tests are heavily concentrated, so
unweighted national averages are biased upward. (5) WorldPop is a modelled surface, not a census,
and is partly built from night-time lights. (6) Boundaries come from geoBoundaries, not from the
national mapping authority. (7) This is **experimental statistics**, not an official indicator,
unless validated against operator or survey data.

**FR.** (1) Les mesures Speedtest sont lancées par les utilisateurs et auto-sélectionnées : ni
échantillon probabiliste, ni pondération de sondage. (2) L'absence de mesure n'est pas l'absence de
service — {100 - nat['pop_coverage_pct']:.0f} % de la population vit dans une cellule sans aucun
test sur le trimestre de référence. (3) Les effets du terminal et du forfait ne peuvent être séparés
de la performance du réseau. (4) Seuls {nat['area_coverage_pct']:.2f} % du territoire sont mesurés
et les tests sont très concentrés : les moyennes nationales non pondérées sont biaisées vers le
haut. (5) WorldPop est une surface modélisée, pas un recensement, et repose en partie sur les
lumières nocturnes. (6) Les limites administratives proviennent de geoBoundaries, et non de
l'autorité cartographique nationale. (7) Il s'agit de **statistiques expérimentales**, et non d'un
indicateur officiel, tant qu'elles n'ont pas été validées contre des données d'opérateurs ou
d'enquête.

## Sources and licences · Sources et licences

- **Ookla® Speedtest Open Data** — <https://github.com/teamookla/ookla-open-data> — **CC BY-NC-SA 4.0**
- **WorldPop** {WORLDPOP_YEAR} ({WORLDPOP_RES}, UN-adjusted) — <https://www.worldpop.org> — CC BY 4.0
- **geoBoundaries** (gbOpen) — <https://www.geoboundaries.org> — CC BY 4.0

## Licence of this product · Licence de ce produit

Released under **CC BY-NC-SA 4.0**, inherited from the Ookla licence (share-alike).
**Non-commercial use only.** Ookla trademarks are the property of Ookla, LLC. This product is not
endorsed by or affiliated with Ookla.

*Diffusé sous licence **CC BY-NC-SA 4.0**, héritée de la licence Ookla (partage dans les mêmes
conditions). **Usage non commercial uniquement.** Les marques Ookla sont la propriété d'Ookla, LLC.
Ce produit n'est ni approuvé par Ookla ni affilié à Ookla.*

## Reproducing · Reproduire

Open the notebook `02-Lab-Ookla-Speedtest-Open-Data-and-WorldPop.ipynb`, set
`COUNTRY_ISO3 = "{COUNTRY_ISO3}"` in the configuration cell, and run all cells. It runs unchanged in
Google Colab, Kaggle and locally, and requires no API key.
"""

LICENSE_TXT = """This work is licensed under the Creative Commons
Attribution-NonCommercial-ShareAlike 4.0 International License (CC BY-NC-SA 4.0).

Full text: https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode

The share-alike obligation is inherited from Ookla Speedtest Open Data, which is
distributed under CC BY-NC-SA 4.0 and from which this work is derived.

Contains information from Ookla Speedtest Open Data (c) Ookla, LLC.
Ookla trademarks are the property of Ookla, LLC. This product is not endorsed
by or affiliated with Ookla.

Population data (c) WorldPop, University of Southampton (CC BY 4.0).
Administrative boundaries (c) geoBoundaries, W. M. Geolab (CC BY 4.0).
"""

(out / "README.md").write_text(README, encoding="utf-8")
(out / "LICENSE").write_text(LICENSE_TXT, encoding="utf-8")
print(f"Écrits : README.md ({len(README):,} caractères, bilingue) · LICENSE")

# 17 · Publication — un jeton demandé, une URL publique obtenue

Le tableau de bord est un unique `index.html` autonome : n'importe quel hébergeur statique le sert.
Nous utilisons **GitHub Pages** : gratuit, permanent, versionné, et sans serveur à maintenir.

**Il n'y a rien à configurer.** Exécutez les deux cellules ci-dessous. La seconde demande un jeton
GitHub dans une invite masquée, puis fait le reste toute seule : elle crée le dépôt **public**,
téléverse tous les fichiers, active Pages, attend la fin de la première construction et affiche
l'URL en ligne.

## Obtenir un jeton — 60 secondes, une seule fois

1. GitHub → votre avatar → **Settings** → **Developer settings** → **Personal access tokens** →
   **Tokens (classic)** → *Generate new token (classic)*.
2. Cochez la portée **`repo`**. Fixez une expiration ; 30 jours suffisent largement.
3. Copiez le jeton. GitHub ne l'affiche **qu'une seule fois**.

Un jeton à portée fine fonctionne aussi : **Repository access → All repositories**, avec
**Read and write** sur *Contents*, *Pages* et *Administration*.

> **Le jeton n'est jamais inscrit dans ce notebook.** Il est lu depuis une invite masquée
> `getpass`, ou depuis la variable d'environnement `GITHUB_TOKEN`, ou — le plus propre sous Colab —
> depuis le panneau **Secrets** (l'icône en forme de clé dans la barre latérale gauche ; nommez le
> secret `GITHUB_TOKEN`). Rien n'est affiché, rien n'est stocké dans une cellule, rien ne se
> retrouve dans le `.ipynb` que vous committerez.

## Ce que « public » signifie ici

Le dépôt est créé **public**, et Pages sert alors le tableau de bord à qui veut, sans
authentification et sans compte GitHub. C'est l'objectif — et c'est aussi le risque. La première
cellule affiche la liste exacte des fichiers sur le point de devenir lisibles par tous : vérifiez-la.
Elle ne doit contenir aucune microdonnée, aucune donnée personnelle et rien sous embargo.

In [ ]:
# Un fichier .nojekyll indique à GitHub Pages de servir le dossier tel quel,
# au lieu de le passer par Jekyll, qui ignore silencieusement certains fichiers.
(out / ".nojekyll").write_text("", encoding="utf-8")

PUBLISH_FILES = sorted(p for p in out.iterdir() if p.is_file())
total_mb = sum(p.stat().st_size for p in PUBLISH_FILES) / 1e6

print(f"{len(PUBLISH_FILES)} fichiers vont devenir publiquement lisibles  ·  {total_mb:.1f} Mo au total\n")
for p in PUBLISH_FILES:
    flag = "   <-- c'est le site web" if p.name == "index.html" else ""
    print(f"  {p.stat().st_size/1024:9,.0f} KB   {p.name}{flag}")

if total_mb > 90:
    callout("Plus de 90 Mo. GitHub refuse les fichiers de plus de 100 Mo et les sites Pages de plus "
            "de 1 Go. Baissez <code>MAX_MAP_TILES</code> dans la cellule de configuration et "
            "reconstruisez le tableau de bord, ou excluez le parquet détaillé de la publication.", "warn")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Publication sur GitHub Pages
# ══════════════════════════════════════════════════════════════════════════════
import base64, getpass

GH_API = "https://api.github.com"
GITHUB_REPO = f"connectivity-{COUNTRY_ISO3.lower()}"   # dépôt qui sera créé
GITHUB_OWNER = ""                                      # vide = votre propre compte


def _gh(method, path, token, **kw):
    return requests.request(
        method, GH_API + path, timeout=90,
        headers={"Authorization": f"Bearer {token}",
                 "Accept": "application/vnd.github+json",
                 "X-GitHub-Api-Version": "2022-11-28"}, **kw)


def _as_token(value):
    """Normalise en chaine propre tout ce qu'un coffre a secrets a pu renvoyer.

    userdata.get() de Colab ne renvoie pas toujours une simple chaîne : selon la
    version du runtime, cela peut être un dictionnaire. Ne rien supposer ; normaliser.
    """
    if value is None:
        return ""
    if isinstance(value, dict):
        for k in ("value", "token", "secret", "GITHUB_TOKEN", "data"):
            if k in value:
                value = value[k]
                break
        else:
            value = next(iter(value.values()), "")
    if isinstance(value, (bytes, bytearray)):
        value = value.decode("utf-8", "ignore")
    return str(value).strip()


def _plausible(tok):
    """Contrôle simple : un jeton GitHub est un mot long, jamais une phrase."""
    return bool(tok) and len(tok) >= 20 and not any(c.isspace() for c in tok)


def _masked(tok):
    return f"{tok[:4]}{'•' * 8}{tok[-4:]}" if len(tok) > 12 else "•" * len(tok)


def ask_token():
    """Environnement, puis secrets Colab, puis invite masquée. Jamais affiché, jamais stocké."""
    # 1 · variable d'environnement
    tok = _as_token(os.environ.get("GITHUB_TOKEN"))
    if _plausible(tok):
        print(f"Jeton lu depuis la variable d'environnement GITHUB_TOKEN ({_masked(tok)}).")
        return tok

    # 2 · panneau Secrets de Colab
    if IN_COLAB:
        try:
            from google.colab import userdata
            tok = _as_token(userdata.get("GITHUB_TOKEN"))
            if _plausible(tok):
                print(f"Jeton lu depuis le panneau Secrets de Colab ({_masked(tok)}).")
                return tok
            if tok:
                print("Le secret Colab GITHUB_TOKEN ne ressemble pas à un jeton — il est ignoré.")
        except Exception as e:
            if "SecretNotFound" not in type(e).__name__:
                print(f"Secrets Colab indisponibles ({type(e).__name__}) — repli sur l'invite de saisie.")

    # 3 · invite masquée, deux tentatives
    for attempt in (1, 2):
        tok = _as_token(getpass.getpass("Jeton GitHub (saisie masquée — rien n'est stocké) : "))
        if _plausible(tok):
            print(f"Jeton reçu ({_masked(tok)}).")
            return tok
        if attempt == 1:
            print("Cela ne ressemble pas à un jeton GitHub (trop court, ou il contient une espace). "
                  "Collez-le à nouveau — il commence par ghp_ ou github_pat_.")
    return ""


def publish_to_github(repo_name=None, owner=None, token=None, files=None,
                      branch="main", private=False, wait=True):
    """Cree le depot public, televerse les sorties, active Pages, renvoie l'URL en ligne."""
    repo_name = repo_name or GITHUB_REPO
    owner = owner or GITHUB_OWNER
    files = files or PUBLISH_FILES
    token = _as_token(token) or ask_token()
    if not token:
        raise RuntimeError("Aucun jeton fourni — publication annulée.")

    # 1 · authentification ------------------------------------------------------
    me = _gh("GET", "/user", token)
    if me.status_code != 200:
        raise RuntimeError(f"Échec de l'authentification ({me.status_code}). Vérifiez le jeton, sa "
                           "date d'expiration, et que la portée « repo » est bien cochée.")
    login = me.json()["login"]
    owner = owner or login
    print(f"Authentifié en tant que {login}" + (f" · publication sous {owner}" if owner != login else ""))

    # 2 · depot --------------------------------------------------------
    r = _gh("GET", f"/repos/{owner}/{repo_name}", token)
    if r.status_code == 200:
        branch = r.json().get("default_branch", branch)
        if r.json().get("private"):
            callout("Ce dépôt est <b>privé</b>. GitHub Pages ne le servira pas publiquement avec une "
                    "offre gratuite — rendez-le public dans Settings, ou choisissez un autre nom.", "risk")
        print(f"Le dépôt {owner}/{repo_name} existe déjà — mise à jour (branche « {branch} »).")
    else:
        body = {"name": repo_name, "private": private, "auto_init": True,
                "description": (f"{COUNTRY_NAME} : indicateurs de connectivité pondérés par la "
                                f"population, données ouvertes Ookla Speedtest et WorldPop, "
                                f"{LATEST_Y} T{LATEST_Q} — FR/EN"),
                "homepage": f"https://{owner}.github.io/{repo_name}/"}
        r = (_gh("POST", "/user/repos", token, json=body) if owner == login
             else _gh("POST", f"/orgs/{owner}/repos", token, json=body))
        if r.status_code not in (200, 201):
            raise RuntimeError(f"Création du dépôt impossible ({r.status_code}) : "
                               f"{r.json().get('message', r.text)}")
        branch = r.json().get("default_branch", branch)
        print(f"Dépôt créé : {owner}/{repo_name} "
              f"({'privé' if private else 'PUBLIC'}, branche par défaut « {branch} »)")
        time.sleep(2)

    # 3 · téléversement ------------------------------------------------------------
    print(f"\nTéléversement de {len(files)} fichiers …")
    uploaded = 0
    for f in files:
        if f.stat().st_size > 95e6:
            print(f"  IGNORÉ {f.name} — au-delà de la limite de 100 Mo de l'API Contents")
            continue
        sha = None
        g = _gh("GET", f"/repos/{owner}/{repo_name}/contents/{f.name}?ref={branch}", token)
        if g.status_code == 200 and isinstance(g.json(), dict):
            sha = g.json().get("sha")
        payload = {"message": f"{'Mise à jour' if sha else 'Ajout'} {f.name}",
                   "content": base64.b64encode(f.read_bytes()).decode(), "branch": branch}
        if sha:
            payload["sha"] = sha
        u = _gh("PUT", f"/repos/{owner}/{repo_name}/contents/{f.name}", token, json=payload)
        ok = u.status_code in (200, 201)
        uploaded += ok
        print(f"  {'OK  ' if ok else 'FAIL'} {f.name:<44} {f.stat().st_size/1024:8,.0f} KB"
              + ("" if ok else f"   -> {u.json().get('message', u.status_code)}"))
    print(f"{uploaded}/{len(files)} fichiers téléversés.")

    # 4 · GitHub Pages ------------------------------------------------------
    print("\nActivation de GitHub Pages …")
    p = _gh("POST", f"/repos/{owner}/{repo_name}/pages", token,
            json={"source": {"branch": branch, "path": "/"}})
    if p.status_code in (201, 204):
        print("  Pages activé.")
    elif p.status_code == 409:
        print("  Pages était déjà activé — le site se reconstruira automatiquement.")
    else:
        _gh("PUT", f"/repos/{owner}/{repo_name}/pages", token,
            json={"source": {"branch": branch, "path": "/"}})
        print(f"  L'API Pages a répondu {p.status_code}. Si l'URL renvoie 404, activez-le à la main : "
              "Settings → Pages → Deploy from a branch → main / root.")

    # 5 · attente de la première construction ------------------------------------------
    site = f"https://{owner}.github.io/{repo_name}/"
    info = _gh("GET", f"/repos/{owner}/{repo_name}/pages", token)
    if info.status_code == 200:
        site = info.json().get("html_url", site)
    if wait:
        print("\nAttente de la première construction (jusqu'à 2 minutes) …")
        for _ in range(24):
            time.sleep(5)
            b = _gh("GET", f"/repos/{owner}/{repo_name}/pages/builds/latest", token)
            status = b.json().get("status") if b.status_code == 200 else None
            if status == "built":
                print("  Construit et servi.")
                break
            if status == "errored":
                print("  La construction a échoué — vérifiez Settings → Pages dans le dépôt.")
                break
        else:
            print("  Construction en cours. L'URL répondra sous peu ; rechargez si elle renvoie 404.")
    return site, f"https://github.com/{owner}/{repo_name}"


print("Prêt. Exécutez la cellule suivante pour publier.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Publier. Cette cellule demande le jeton et fait tout le reste elle-même.
# ══════════════════════════════════════════════════════════════════════════════
try:
    SITE_URL, REPO_URL = publish_to_github()
    display(HTML(f"""
    <div style="font-family:{FONT};background:linear-gradient(120deg,{FOREST},{DEEP} 45%,{GREEN});
                color:#fff;border-radius:6px;padding:26px 30px;margin-top:14px">
      <div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:{GOLD}">
        PUBLIÉ · ACCESSIBLE PUBLIQUEMENT · FR / EN</div>
      <div style="font-size:23px;font-weight:700;margin-top:8px">Votre tableau de bord est en ligne</div>
      <div style="margin-top:15px;font-size:15px">
        <a href="{SITE_URL}" target="_blank" style="color:#fff;background:rgba(255,255,255,.17);
           padding:10px 17px;border-radius:4px;text-decoration:none;font-weight:700">{SITE_URL}</a>
      </div>
      <div style="font-size:12.5px;color:#E6F6EE;margin-top:15px;line-height:1.6">
        Dépôt : <a href="{REPO_URL}" target="_blank" style="color:{GOLD}">{REPO_URL}</a><br>
        N'importe qui peut ouvrir ce lien — sans compte, sans authentification. Relancez cette
        cellule à tout moment pour mettre à jour le site ; le dépôt conserve chaque version publiée.
      </div>
      <div style="height:4px;background:{GOLD};margin-top:20px;width:110px"></div>
    </div>"""))
    print(f"\nURL publique : {SITE_URL}")

except Exception as e:
    SITE_URL = None
    callout(f"<b>La publication n'a pas abouti.</b> {type(e).__name__} : {e}<br><br>"
            "Causes les plus fréquentes : jeton expiré, portée <code>repo</code> non cochée, nom "
            "de dépôt déjà pris par un autre projet, ou notebook exécuté sans invite interactive "
            "(nbconvert, intégration continue). Dans ce dernier cas, définissez la variable "
            "d'environnement <code>GITHUB_TOKEN</code> avant l'exécution. "
            "La voie manuelle ci-dessous fonctionne toujours.", "warn")

## Si vous préférez pousser vous-même

```bash
cd outputs
git init && git add -A && git commit -m "Tableau de bord connectivité — Ookla x WorldPop (FR/EN)"
git branch -M main
git remote add origin https://github.com/<votre-compte>/<votre-depot>.git
git push -u origin main
```

Puis, dans le dépôt : **Settings → Pages → Source : *Deploy from a branch* → `main` / `/ (root)`
→ Save**. Une minute plus tard environ, le tableau de bord est en ligne à l'adresse
`https://<votre-compte>.github.io/<votre-depot>/` — parce que le fichier a délibérément été nommé
`index.html`, qui est ce qu'un serveur web renvoie à la racine d'un site.

Pas de ligne de commande du tout ? Sur github.com : **New repository** → cochez **Public** →
*uploading an existing file* → glissez-déposez tout le dossier `outputs` → *Commit changes* → puis
**Settings → Pages** comme ci-dessus.

## Vérifier que c'est réellement public

Ouvrez l'URL dans une **fenêtre de navigation privée**, ou envoyez-la à quelqu'un qui n'est pas
connecté à GitHub. Si elle s'affiche là, et que le sélecteur FR / EN fonctionne, c'est bien public.
Trois choses cassent cela en pratique :

- le dépôt est **privé** — Pages exige alors une offre payante et renvoie 404 sur l'offre gratuite ;
- **Settings → Pages** n'a jamais été enregistré, aucun site n'a donc jamais été construit ;
- le fichier d'entrée ne s'appelle pas `index.html`, l'URL racine n'a donc rien à servir.

## Deux choses à faire avant de pousser

1. **Faites valider la licence par votre service juridique.** CC BY-NC-SA est inhabituelle pour un
   portail statistique public, et la clause non commerciale voyage avec toute œuvre dérivée.
2. **Lisez votre propre déclaration de limites à voix haute, dans les deux langues.** Si vous ne
   seriez pas à l'aise pour la défendre devant un journaliste, elle n'est pas terminée.

# 18 · Exercices

Faites-les dans l'ordre ; chacun prend de 5 à 15 minutes et chacun modifie un chiffre que vous venez
de publier.

**1 · Changez de pays.** Mettez un pays voisin dans `COUNTRY_ISO3` et relancez tout. Comparez la
couverture de la mesure en population. Pourquoi diffère-t-elle autant entre deux pays de taille
comparable ?

**2 · Cassez la pondération volontairement.** Remplacez `d_median_pop` par `d_median_tile` dans le
graphique de classement. Quels districts bougent, et dans quel sens ? Rédigez une phrase expliquant
ce mouvement à une personne non statisticienne.

**3 · Changez le dénominateur.** L'indicateur `pct_above_bb` est calculé sur la population
*mesurée*. Recalculez-le sur la population *totale*, en traitant les personnes non mesurées d'abord
comme inconnues, puis comme nulles. Vous disposez maintenant de trois chiffres légitimes. Lequel
publieriez-vous, et que dirait exactement la note de bas de page ?

**4 · Testez les seuils d'urbanisation.** Faites passer `URBAN_DENS_MIN` de 1500 à 1000, puis à
2500. Quelle part de l'écart urbain–rural relève des données, et quelle part relève de votre seuil ?

**5 · Ajoutez une carte d'évolution trimestrielle.** Calculez, pour chaque unité administrative,
l'écart de débit médian pondéré entre le trimestre le plus ancien et le plus récent, et
cartographiez-le avec une palette divergente (brique → blanc → vert). Attention : une unité qui
gagne des *carreaux* peut perdre du *débit* par simple effet de composition.

**6 · Passez au 100 m.** Mettez `WORLDPOP_RES = "100m"`. La cellule de population est désormais plus
petite que le carreau Ookla : la règle de répartition s'inverse. La médiane nationale pondérée
bouge-t-elle de plus de 1 Mbit/s ? Si non, vous venez de justifier l'usage de la grille de 1 km en
production.

**7 · Confrontez à des données officielles.** Joignez votre tableau aux chiffres de l'UIT, d'un
opérateur ou du recensement dont vous disposez déjà pour deux ou trois régions. Sont-elles classées
dans le même ordre ? Une corrélation de rangs est un test suffisant pour un indicateur expérimental ;
l'accord des niveaux n'est pas exigé et ne doit pas être attendu.

**8 · Préparez le paragraphe honnête.** Rédigez les trois phrases que vous diriez si un journaliste
vous demandait « alors, quel est le débit internet moyen dans mon district ? ». Elles ont leur place
dans votre README, pas dans votre tête.

# 19 · Sources

| Source | Référence | Licence |
|---|---|---|
| **Ookla® Speedtest Open Data** | `github.com/teamookla/ookla-open-data` — carreaux de performance mondiaux, trimestriels depuis le T1 2019, Web-Mercator z16 | **CC BY-NC-SA 4.0** |
| **WorldPop** | `worldpop.org` — grille mondiale de population 2000–2020, ajustée ONU, université de Southampton | CC BY 4.0 |
| **geoBoundaries** | `geoboundaries.org` — diffusion gbOpen, W. M. Geolab, William & Mary | CC BY 4.0 |
| **DuckDB** | `duckdb.org` — base analytique en processus, extension `httpfs` pour le parquet distant | MIT |
| **UIT** | *Measuring digital development: Facts and Figures* — seuils de référence du haut débit utilisable | — |

**Attribution obligatoire dans toute publication dérivée de ce notebook :**

> Contient des informations issues des données ouvertes Ookla® Speedtest, © Ookla, LLC, utilisées
> sous licence CC BY-NC-SA 4.0. Les marques Ookla sont la propriété d'Ookla, LLC. Ce produit n'est
> ni approuvé par Ookla ni affilié à Ookla. Données de population © WorldPop (CC BY 4.0). Limites
> administratives © geoBoundaries (CC BY 4.0).

---

<div style="background:linear-gradient(120deg,#00553A 0%,#00704A 45%,#00A86A 100%);
            padding:26px 30px;border-radius:6px;color:#fff;font-family:Calibri,sans-serif">
  <div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#F5C242">FIN DU LABORATOIRE</div>
  <div style="font-size:24px;font-weight:700;margin-top:7px">Vous disposez maintenant d'un produit statistique public.</div>
  <div style="font-size:13.5px;color:#E6F6EE;margin-top:9px;line-height:1.6">
    Une URL publique en ligne, une méthode documentée, une déclaration explicite des limites, et une
    licence qui résiste à un examen juridique. Relancez le notebook avec un autre
    <code>COUNTRY_ISO3</code> et l'ensemble du produit est reconstruit.
  </div>
  <div style="height:4px;background:#F5C242;margin-top:20px;width:110px"></div>
</div>